# 화장품 성분 기능 분류 모델 학습 및 실험 기록

CosIng의 성분명과 화학적 설명을 입력으로 사용하여 6개 기능을 다중 라벨 분류합니다. Logistic Regression과 Linear SVM을 직접 학습하고, 입력 표현·불균형 보정·기준값·규제 강도를 비교했습니다.

## 읽는 순서

1. 데이터 확인 및 전처리
2. 데이터 분할과 기능 선정
3. 기준 실험: 성분명 TF-IDF + LR / SVM
4. 클래스 불균형 보정과 초기 오류 분석
5. 입력 개선: 성분명 + 설명
6. 표현 방식 비교: MiniLM
7. SVM 기준값 및 규제 강도 탐색
8. 최종 모델 평가와 저장
9. 모델 추론 확인과 초기 시연 사례
10. 부록: 초기 배포 코드 생성 기록

> **보존 범위:** 기존 코드의 순서와 저장된 출력은 유지했습니다. 아래 출력은 기존 실행 기록이며, 이 정리 작업에서 모델을 재학습하지 않았습니다. 처음부터 실행하려면 Colab에서 CSV 업로드부터 순서대로 실행하세요. MiniLM 변환은 CPU에서 오래 걸릴 수 있습니다.
>
> **평가 해석:** 실험 과정에서 테스트셋을 여러 번 확인했습니다. 따라서 최종 테스트 성능을 완전히 미공개인 데이터에 대한 단 한 번의 평가로 서술하지 않습니다. 추가 연구에서는 새로운 홀드아웃 데이터로 평가해야 합니다.
>
> **서비스 코드:** 부록은 초기 버전의 생성 기록입니다. 현재 OCR·추천 UI를 포함한 최종 서비스 소스는 별도의 `index.html`, `api/predict.py`, `requirements.txt`를 기준으로 합니다.

> **코드 정리:** 반복 평가·학습 코드를 공통 함수로 통합했습니다. 기존 출력은 비교 참고를 위해 그대로 보존했으며 수정 코드 전체를 재학습 실행한 결과는 아닙니다. 제출 전에 Colab에서 위에서부터 재실행해 코드와 출력의 일치를 확인하세요.


## 1. 데이터 확인 및 전처리

원본 확인, 결측치 처리, 이름 정규화, 기능 라벨 분리 및 중복 통합 과정을 확인합니다.


### CSV 업로드 (데이터: 2010-10-15 ~ 2023-06-13)


In [3]:
from google.colab import files

uploaded = files.upload()

Saving COSING_Ingredients-Fragrance%20Inventory_v2.csv to COSING_Ingredients-Fragrance%20Inventory_v2.csv


### 데이터 불러오기


In [5]:
import io
import pandas as pd

filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("데이터 크기:", df.shape)
print("컬럼 목록:", df.columns.tolist())

df.head()

데이터 크기: (30080, 10)
컬럼 목록: ['COSING Ref No', 'INCI name', 'INN name', 'Ph. Eur. Name', 'CAS No', 'EC No', 'Chem/IUPAC Name / Description', 'Restriction', 'Function', 'Update Date']


,COSING Ref No,INCI name,INN name,Ph. Eur. Name,CAS No,EC No,Chem/IUPAC Name / Description,Restriction,Function,Update Date
0,94753,DISODIUM TETRAMETHYLHEXADECENYLCYSTEINE FORM...,NaN,NaN,"2040469-40-5, 2422121-34-2",,Disodium Tetramethylhexadecenylcysteine Formyl...,NaN,SKIN PROTECTING,16/06/2020
1,99268,(ANGELICA ACUTILOBA/PAEONIA LACTIFLORA) ROOT/...,NaN,NaN,,,(Angelica Acutiloba/Paeonia Lactiflora) Root/C...,NaN,SKIN CONDITIONING - EMOLLIENT,19/03/2021
2,99270,(ARCTIUM LAPPA/RAPHANUS SATIVUS/MALVA VERTICI...,NaN,NaN,,,(Arctium Lappa/Raphanus Sativus/Malva Verticil...,NaN,SKIN CONDITIONING - EMOLLIENT,19/03/2021
3,94896,(LIQUIDAMBAR STYRACIFLUA/TRIBULUS TERRESTRIS)...,NaN,NaN,,,(Liquidambar Styraciflua/Tribulus Terrestris)...,NaN,SKIN CONDITIONING,01/12/2017
4,95645,ACRYLATES/VA/VINYL NEODECANOATE COPOLYMER,NaN,NaN,99728-55-9,,Acrylates/VA/Vinyl Neodecanoate Copolymer is ...,NaN,PLASTICISER,14/02/2018


### 공백 정리 및 결측치 확인


In [6]:
# 문자열 앞뒤 공백 제거
for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].str.strip()

# 빈 문자열을 결측치로 통일
df = df.replace(r"^\s*$", pd.NA, regex=True)

df.isna().sum()

,0
COSING Ref No,0
INCI name,0
INN name,29561
Ph. Eur. Name,29897
CAS No,11602
EC No,14947
Chem/IUPAC Name / Description,476
Restriction,28114
Function,216
Update Date,0


### 학습에 사용할 열 선택 및 기능 라벨 정리


In [7]:
# 성분명과 기능 열만 선택
data = df[["INCI name", "Function"]].copy()

# 성분명 또는 기능이 비어 있는 행 제거
data = data.dropna(subset=["INCI name", "Function"])

# 성분명 표기 통일
data["INCI name"] = (
    data["INCI name"]
    .str.normalize("NFKC")
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# 쉼표로 구분된 기능을 리스트로 변환
def split_functions(value):
    return sorted({
        label.strip()
        for label in value.split(",")
        if label.strip() and label.strip() != "NOT REPORTED"
    })

data["labels"] = data["Function"].apply(split_functions)

# 실제 기능이 남아 있는 행만 유지
data = data[data["labels"].str.len() > 0].copy()

print("기능 라벨 정리 후 행 수:", len(data))
data[["INCI name", "labels"]].head()

기능 라벨 정리 후 행 수: 29308


,INCI name,labels
0,DISODIUM TETRAMETHYLHEXADECENYLCYSTEINE FORMYL...,[SKIN PROTECTING]
1,(ANGELICA ACUTILOBA/PAEONIA LACTIFLORA) ROOT/C...,[SKIN CONDITIONING - EMOLLIENT]
2,(ARCTIUM LAPPA/RAPHANUS SATIVUS/MALVA VERTICIL...,[SKIN CONDITIONING - EMOLLIENT]
3,(LIQUIDAMBAR STYRACIFLUA/TRIBULUS TERRESTRIS) ...,[SKIN CONDITIONING]
4,ACRYLATES/VA/VINYL NEODECANOATE COPOLYMER,[PLASTICISER]


### 중복 성분명 통합


In [8]:
data = (
    data.groupby("INCI name", as_index=False)
    .agg({
        "labels": lambda lists: sorted({
            label
            for label_list in lists
            for label in label_list
        })
    })
)

print("중복 통합 후 성분 수:", len(data))
print("성분명 중복 수:", data["INCI name"].duplicated().sum())

data.head()

중복 통합 후 성분 수: 29307
성분명 중복 수: 0


,INCI name,labels
0,(20S)-PROTOPANAXADIOL,"[SKIN CONDITIONING, SKIN CONDITIONING - EMOLLI..."
1,(ADONIS AMURENSIS/AKEBIA QUINATA/ALBIZIA JULIB...,"[SKIN CONDITIONING - HUMECTANT, SKIN PROTECTING]"
2,(AGARICUS BLAZEI FERMENT + GANODERMA LUCIDUM F...,"[ANTI-SEBUM, ANTIOXIDANT, SKIN PROTECTING]"
3,(ALPINIA ZERUMBET/ARTEMISIA PRINCEPS/AVERRHOA ...,[SKIN CONDITIONING - MISCELLANEOUS]
4,(ANGELICA ACUTILOBA/PAEONIA LACTIFLORA) ROOT/C...,[SKIN CONDITIONING - EMOLLIENT]


## 2. 데이터 분할과 기능 선정

고정 seed로 학습·검증·테스트를 분할하고 기능별 분포 및 0·1 정답표를 확인합니다. 희귀 기능까지 포함한 6개 기능의 최종 선정은 서비스 목적에 따른 수동 결정입니다.


### 학습·검증·테스트 데이터 분할
(평가용 데이터를 넉넉히 확보하기 위해 70, 15, 15 비율로 분할함.)


In [9]:
from sklearn.model_selection import train_test_split

# 실행할 때마다 동일하게 나뉘도록 설정
SEED = 42

# 전체의 70%는 학습, 30%는 임시 데이터로 분리
train_data, temp_data = train_test_split(
    data,
    test_size=0.30,
    random_state=SEED
)

# 임시 데이터를 절반씩 나눠 검증 15%, 테스트 15% 구성
val_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=SEED
)

# 인덱스 정리
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print("학습 데이터:", len(train_data))
print("검증 데이터:", len(val_data))
print("테스트 데이터:", len(test_data))

# 데이터 간 같은 성분명이 겹치지 않는지 확인
train_names = set(train_data["INCI name"])
val_names = set(val_data["INCI name"])
test_names = set(test_data["INCI name"])

assert train_names.isdisjoint(val_names)
assert train_names.isdisjoint(test_names)
assert val_names.isdisjoint(test_names)

print("성분명 중복 없이 분할 완료!")

학습 데이터: 20514
검증 데이터: 4396
테스트 데이터: 4397
성분명 중복 없이 분할 완료!


### 학습 데이터에서 빈도가 높은 기능 35개 확인


In [10]:
from collections import Counter

# 학습 데이터의 기능별 등장 횟수 집계
label_counts = Counter(
    label
    for labels in train_data["labels"]
    for label in labels
)

# 등장 횟수가 많은 순으로 35개 선택
# 횟수가 같으면 기능명 순으로 정렬
top_labels = [
    label
    for label, count in sorted(
        label_counts.items(),
        key=lambda item: (-item[1], item[0])
    )[:35]
]

print("선택한 기능 수:", len(top_labels))

# 선택한 기능과 학습 데이터 내 개수 확인
label_summary = pd.DataFrame({
    "기능": top_labels,
    "학습 데이터 내 개수": [label_counts[label] for label in top_labels]
})

display(label_summary)

선택한 기능 수: 35


,기능,학습 데이터 내 개수
0,SKIN CONDITIONING,9059
1,SKIN CONDITIONING - EMOLLIENT,2338
2,HAIR CONDITIONING,2299
3,SURFACTANT - CLEANSING,2160
4,SURFACTANT - EMULSIFYING,1924
5,PERFUMING,1889
6,ANTIOXIDANT,1825
7,SKIN PROTECTING,1633
8,HUMECTANT,1454
9,FRAGRANCE,1004


### 상위 35개 중 피부 타입별 분석에 사용할 기능 6개 선정 (학습 데이터의 기능별 빈도를 집계하여 상위 35개 기능을 확인하였다. 이후 얼굴용 스킨케어 분석이라는 서비스 목적, 기능의 해석 가능성 및 구현 범위를 고려하여 제작자가 최종 6개 기능을 선정하였다.)


In [11]:
selected_functions = [
    "SKIN CONDITIONING - HUMECTANT",
    "SKIN CONDITIONING - EMOLLIENT",
    "ANTI-SEBUM",
    "SKIN PROTECTING",
    "SKIN CONDITIONING - OCCLUSIVE",
    "SOOTHING"
]

function_names = {
    "SKIN CONDITIONING - HUMECTANT": "피부 보습",
    "SKIN CONDITIONING - EMOLLIENT": "피부 유연화",
    "ANTI-SEBUM": "피지 조절",
    "SKIN PROTECTING": "피부 보호",
    "SKIN CONDITIONING - OCCLUSIVE": "수분 증발 억제",
    "SOOTHING": "피부·두피 불편감 완화"
}

# 학습 데이터 기준으로 기능별 개수 확인
selected_summary = pd.DataFrame({
    "기능": selected_functions,
    "설명": [function_names[label] for label in selected_functions],
    "학습 데이터 내 개수": [
        label_counts[label] for label in selected_functions
    ],
    "상위 35개 포함": [
        label in top_labels for label in selected_functions
    ]
})

print("최종 선정한 기능 수:", len(selected_functions))
display(selected_summary)

최종 선정한 기능 수: 6


,기능,설명,학습 데이터 내 개수,상위 35개 포함
0,SKIN CONDITIONING - HUMECTANT,피부 보습,157,True
1,SKIN CONDITIONING - EMOLLIENT,피부 유연화,2338,True
2,ANTI-SEBUM,피지 조절,184,True
3,SKIN PROTECTING,피부 보호,1633,True
4,SKIN CONDITIONING - OCCLUSIVE,수분 증발 억제,15,False
5,SOOTHING,피부·두피 불편감 완화,88,False


### 선정한 기능을 0·1 정답표로 변환


In [15]:
from sklearn.preprocessing import MultiLabelBinarizer

selected_labels = set(selected_functions)

# 성분별 기능 중 선정한 6개만 남기기
def keep_selected_labels(label_lists):
    return [
        [label for label in labels if label in selected_labels]
        for labels in label_lists
    ]

# 기능 순서를 고정하여 정답 배열 생성
mlb = MultiLabelBinarizer(classes=selected_functions)

y_train = mlb.fit_transform(
    keep_selected_labels(train_data["labels"])
)

y_val = mlb.transform(
    keep_selected_labels(val_data["labels"])
)

y_test = mlb.transform(
    keep_selected_labels(test_data["labels"])
)

print("학습 정답 크기:", y_train.shape)
print("검증 정답 크기:", y_val.shape)
print("테스트 정답 크기:", y_test.shape)

# 1. 정답 배열을 표로 만들기
answer_table = pd.DataFrame(
    y_train,
    columns=mlb.classes_
)

# 2. 성분명 추가
answer_table.insert(
    0,
    "성분명",
    train_data["INCI name"].to_numpy()
)

# 3. 선택한 기능이 하나 이상 기록된 성분 찾기
has_selected_function = y_train.sum(axis=1) > 0

# 4. 해당 성분의 정답표 출력
display(
    answer_table.loc[has_selected_function].head(10)
)

print("선택한 기능이 있는 학습 성분 수:",
      int(has_selected_function.sum()))

print("선택한 기능이 모두 0인 학습 성분 수:",
      int((~has_selected_function).sum()))

학습 정답 크기: (20514, 6)
검증 정답 크기: (4396, 6)
테스트 정답 크기: (4397, 6)


,성분명,SKIN CONDITIONING - HUMECTANT,SKIN CONDITIONING - EMOLLIENT,ANTI-SEBUM,SKIN PROTECTING,SKIN CONDITIONING - OCCLUSIVE,SOOTHING
25,HYDROLYZED ARTHROSPIRA EXTRACT,0,0,0,1,0,0
29,PPG-20 OLEYL ETHER,0,1,0,0,0,0
31,TEA-PCA,0,0,0,1,0,0
35,RHUS GLABRA BARK/BERRY/ROOT EXTRACT,0,0,1,0,0,0
37,ALLANTOIN CALCIUM PANTOTHENATE,0,0,0,1,0,1
43,LARIX SIBIRICA WOOD EXTRACT,0,0,0,1,0,0
55,LACTOBACILLUS/SKELETONEMA FERMENT,0,0,0,1,0,0
56,TURTLE EXTRACT,0,1,0,0,0,0
57,PETASITES JAPONICUS LEAF EXTRACT,0,1,0,0,0,0
59,SAPINDUS MUKOROSSI SEED OIL,0,1,0,1,0,0


선택한 기능이 있는 학습 성분 수: 4063
선택한 기능이 모두 0인 학습 성분 수: 16451


### 기능별 데이터 분포 확인


In [16]:
split_summary = pd.DataFrame({
    "기능": mlb.classes_,
    "학습 양성 개수": y_train.sum(axis=0),
    "검증 양성 개수": y_val.sum(axis=0),
    "테스트 양성 개수": y_test.sum(axis=0)
})

display(split_summary)

# 기능이 전혀 배정되지 않은 분할이 있는지 확인
count_columns = [
    "학습 양성 개수",
    "검증 양성 개수",
    "테스트 양성 개수"
]

if (split_summary[count_columns] == 0).any().any():
    print("확인 필요: 일부 기능이 특정 분할에 없습니다.")
else:
    print("6개 기능 모두 학습·검증·테스트에 포함되어 있습니다.")

,기능,학습 양성 개수,검증 양성 개수,테스트 양성 개수
0,SKIN CONDITIONING - HUMECTANT,157,37,21
1,SKIN CONDITIONING - EMOLLIENT,2338,495,506
2,ANTI-SEBUM,184,34,52
3,SKIN PROTECTING,1633,383,340
4,SKIN CONDITIONING - OCCLUSIVE,15,9,2
5,SOOTHING,88,18,21


6개 기능 모두 학습·검증·테스트에 포함되어 있습니다.


## 3. 기준 실험: 성분명 TF-IDF + LR / SVM

성분명만 사용한 두 모델의 학습 및 기준값 탐색 기록입니다. 이 절의 ‘최종’이라는 기존 표현은 초기 실험 단계의 결과를 뜻합니다.


### 성분명을 숫자 특징으로 변환 — TF-IDF


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion

# 입력: 성분명
X_train_text = train_data["INCI name"]
X_val_text = val_data["INCI name"]
X_test_text = test_data["INCI name"]

# 단어 패턴과 글자 조각 패턴을 함께 사용
vectorizer = FeatureUnion([
    (
        "word",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=30000,
            sublinear_tf=True
        )
    ),
    (
        "char",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=3,
            max_features=50000,
            sublinear_tf=True
        )
    )
])

# 특징을 만드는 기준은 학습 데이터에서만 학습
X_train = vectorizer.fit_transform(X_train_text)

# 검증·테스트에는 같은 기준을 적용
X_val = vectorizer.transform(X_val_text)
X_test = vectorizer.transform(X_test_text)

print("학습 입력 크기:", X_train.shape)
print("검증 입력 크기:", X_val.shape)
print("테스트 입력 크기:", X_test.shape)

학습 입력 크기: (20514, 47976)
검증 입력 크기: (4396, 47976)
테스트 입력 크기: (4397, 47976)


### 첫 번째 모델 학습 — Logistic Regression


### 공통 학습·평가 도구

중복되던 평가 지표, 기능별 표, 기준값 탐색, 학습 시간 측정을 함수로 묶었습니다.


In [ ]:
# 반복되는 학습·평가·기준값 탐색을 공통 함수로 관리합니다.
import time
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

def evaluate_predictions(y_true, prediction):
    return {
        "Micro F1": f1_score(y_true, prediction, average="micro", zero_division=0),
        "Macro F1": f1_score(y_true, prediction, average="macro", zero_division=0),
        "Micro Precision": precision_score(y_true, prediction, average="micro", zero_division=0),
        "Micro Recall": recall_score(y_true, prediction, average="micro", zero_division=0),
    }

def function_metrics(y_true, prediction, include_count=True):
    report = classification_report(y_true, prediction, target_names=mlb.classes_,
                                   output_dict=True, zero_division=0)
    table = pd.DataFrame(report).T.loc[list(mlb.classes_),
                                    ["precision", "recall", "f1-score", "support"]].copy()
    if include_count:
        table["양성 예측 수"] = prediction.sum(axis=0)
    return table

def select_threshold(y_true, scores, thresholds):
    table = pd.DataFrame([
        {"기준값": threshold, **evaluate_predictions(y_true, (scores >= threshold).astype(int))}
        for threshold in thresholds
    ])
    best = table.loc[table["Micro F1"].idxmax()]
    return table, float(best["기준값"])

def fit_timed(model, X, y):
    start = time.perf_counter()
    model.fit(X, y)
    return time.perf_counter() - start

def fit_variants(X, y):
    models = {name: clone(model) for name, model in {
        "LR 기본": lr_model, "SVM 기본": svm_model,
        "LR balanced": lr_balanced_model, "SVM balanced": svm_balanced_model,
    }.items()}
    times = {}
    for name, model in models.items():
        times[name] = fit_timed(model, X, y)
        print(f"{name} 학습 완료: {times[name]:.2f}초")
    return models, times

def validate_variants(models, X, y, settings, column, value):
    results, rows = {}, []
    for name, model in models.items():
        method, thresholds = settings[name]
        scores = getattr(model, method)(X)
        table, threshold = select_threshold(y, scores, thresholds)
        prediction = (scores >= threshold).astype(int)
        results[name] = {"table": table, "threshold": threshold,
                         "score_method": method, "prediction": prediction}
        rows.append({"설정": name, column: value, "기준값": threshold,
                     **evaluate_predictions(y, prediction)})
    return results, pd.DataFrame(rows)


In [18]:
import time

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

# 모델 생성
lr_model = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        solver="liblinear",
        max_iter=1000,
        random_state=SEED
    ),
    n_jobs=1
)

# 학습 시간 측정
start_time = time.perf_counter()

# 성분명의 숫자 특징과 기능 정답으로 학습
lr_model.fit(X_train, y_train)

lr_training_time = time.perf_counter() - start_time

print("Logistic Regression 학습 완료!")
print(f"학습 시간: {lr_training_time:.2f}초")
print("학습한 기능 수:", len(lr_model.estimators_))

Logistic Regression 학습 완료!
학습 시간: 2.99초
학습한 기능 수: 6


### Logistic Regression의 검증 성능 확인


| 지표 | 의미 |
|---|---|
| **Precision** | 해당 기능이 있다고 예측한 것 중 실제로 기록된 비율 |
| **Recall** | 해당 기능이 실제로 기록된 것 중 모델이 찾아낸 비율 |
| **F1** | Precision과 Recall을 함께 고려한 점수 |
| **support** | 검증 데이터에서 해당 기능이 기록된 성분 수 |
| **Micro F1** | 모든 기능의 예측을 합쳐 계산한 점수 |
| **Macro F1** | 기능별 F1을 동일한 비중으로 평균 낸 점수 |

In [19]:
lr_val_proba = lr_model.predict_proba(X_val)
lr_threshold = 0.5
lr_val_pred = (lr_val_proba >= lr_threshold).astype(int)
lr_val_metrics = evaluate_predictions(y_val, lr_val_pred)
print("Logistic Regression 검증 성능")
display(pd.DataFrame(lr_val_metrics.items(), columns=["평가지표", "점수"]).round(4))
function_report = function_metrics(y_val, lr_val_pred, include_count=False)
display(function_report.round(4))


Logistic Regression 검증 성능


,평가지표,점수
0,Micro F1,0.2720
1,Macro F1,0.0939
2,Micro Precision,0.6627
3,Micro Recall,0.1711


,precision,recall,f1-score,support
SKIN CONDITIONING - HUMECTANT,0.0000,0.0000,0.0000,37.0
SKIN CONDITIONING - EMOLLIENT,0.6467,0.2404,0.3505,495.0
ANTI-SEBUM,0.0000,0.0000,0.0000,34.0
SKIN PROTECTING,0.7059,0.1253,0.2129,383.0
SKIN CONDITIONING - OCCLUSIVE,0.0000,0.0000,0.0000,9.0
SOOTHING,0.0000,0.0000,0.0000,18.0


### 예측 기준값을 바꿔 검증 성능 비교


In [20]:
threshold_table, lr_threshold = select_threshold(
    y_val, lr_val_proba, [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50])
display(threshold_table.round(4))
print("선택한 기준값:", lr_threshold)
lr_val_pred = (lr_val_proba >= lr_threshold).astype(int)
updated_table = function_metrics(y_val, lr_val_pred)
display(updated_table.round(4))


,기준값,Micro F1,Macro F1,Micro Precision,Micro Recall
0,0.05,0.2827,0.2033,0.1708,0.8186
1,0.10,0.3767,0.1964,0.2670,0.6393
2,0.15,0.4126,0.1876,0.3426,0.5184
3,0.20,0.4301,0.1806,0.4239,0.4365
4,0.30,0.4085,0.1410,0.5503,0.3248
5,0.40,0.3449,0.1189,0.6142,0.2398
6,0.50,0.2720,0.0939,0.6627,0.1711


선택한 기준값: 0.2


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.0000,0.0000,0.0000,37.0,0
SKIN CONDITIONING - EMOLLIENT,0.4547,0.5576,0.5009,495.0,607
ANTI-SEBUM,0.4000,0.0588,0.1026,34.0,5
SKIN PROTECTING,0.3760,0.3838,0.3798,383.0,391
SKIN CONDITIONING - OCCLUSIVE,0.0000,0.0000,0.0000,9.0,0
SOOTHING,0.5000,0.0556,0.1000,18.0,2


### 두 번째 모델 학습 — Linear SVM


In [21]:
from sklearn.svm import LinearSVC

svm_model = OneVsRestClassifier(
    LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=SEED
    ),
    n_jobs=1
)

start_time = time.perf_counter()

svm_model.fit(X_train, y_train)

svm_training_time = time.perf_counter() - start_time

print("Linear SVM 학습 완료!")
print(f"학습 시간: {svm_training_time:.2f}초")
print("학습한 기능 수:", len(svm_model.estimators_))

Linear SVM 학습 완료!
학습 시간: 1.65초
학습한 기능 수: 6


### SVM의 기준값별 검증 성능 비교


In [22]:
svm_val_scores = svm_model.decision_function(X_val)
svm_threshold_table, svm_threshold = select_threshold(
    y_val, svm_val_scores, [-0.9, -0.75, -0.5, -0.25, 0.0, 0.25])
display(svm_threshold_table.round(4))
print("선택한 SVM 기준값:", svm_threshold)
svm_val_pred = (svm_val_scores >= svm_threshold).astype(int)
svm_function_table = function_metrics(y_val, svm_val_pred)
display(svm_function_table.round(4))


,기준값,Micro F1,Macro F1,Micro Precision,Micro Recall
0,-0.90,0.3050,0.2026,0.1945,0.7070
1,-0.75,0.3571,0.2622,0.2496,0.6270
2,-0.50,0.4003,0.2953,0.3356,0.4959
3,-0.25,0.4028,0.2278,0.4220,0.3852
4,0.00,0.3751,0.2070,0.5310,0.2900
5,0.25,0.3040,0.1560,0.6352,0.1998


선택한 SVM 기준값: -0.25


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.2000,0.0811,0.1154,37.0,15
SKIN CONDITIONING - EMOLLIENT,0.4635,0.4747,0.4691,495.0,507
ANTI-SEBUM,0.2069,0.1765,0.1905,34.0,29
SKIN PROTECTING,0.3886,0.3368,0.3608,383.0,332
SKIN CONDITIONING - OCCLUSIVE,0.0000,0.0000,0.0000,9.0,0
SOOTHING,0.3750,0.1667,0.2308,18.0,8


### 두 모델의 테스트 성능 최종 비교


In [23]:
# 검증 단계에서 선택한 모델과 기준값 고정
selected_model_name = "Logistic Regression"

# 테스트 예측
lr_test_pred = (
    lr_model.predict_proba(X_test) >= lr_threshold
).astype(int)

svm_test_pred = (
    svm_model.decision_function(X_test) >= svm_threshold
).astype(int)

# 두 모델을 동일한 지표로 평가
test_results = []

for model_name, prediction, training_time in [
    ("Logistic Regression", lr_test_pred, lr_training_time),
    ("Linear SVM", svm_test_pred, svm_training_time)
]:
    test_results.append({
        "모델": model_name,
        "Micro F1": f1_score(
            y_test, prediction, average="micro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_test, prediction, average="macro", zero_division=0
        ),
        "Micro Precision": precision_score(
            y_test, prediction, average="micro", zero_division=0
        ),
        "Micro Recall": recall_score(
            y_test, prediction, average="micro", zero_division=0
        ),
        "학습 시간(초)": training_time
    })

test_comparison = pd.DataFrame(test_results)

print("최종 테스트 성능 비교")
display(test_comparison.round(4))

print("검증 Micro F1 기준 선택 모델:", selected_model_name)
print("Logistic Regression 기준값:", lr_threshold)
print("Linear SVM 기준값:", svm_threshold)

최종 테스트 성능 비교


,모델,Micro F1,Macro F1,Micro Precision,Micro Recall,학습 시간(초)
0,Logistic Regression,0.4441,0.1562,0.4259,0.4639,2.9894
1,Linear SVM,0.4253,0.2999,0.4422,0.4098,1.6507


검증 Micro F1 기준 선택 모델: Logistic Regression
Logistic Regression 기준값: 0.2
Linear SVM 기준값: -0.25


### 선택 모델의 기능별 테스트 성능 확인


In [24]:
final_function_table = function_metrics(y_test, lr_test_pred)
display(final_function_table.round(4))


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.0000,0.0000,0.0000,21.0,1
SKIN CONDITIONING - EMOLLIENT,0.4595,0.5949,0.5185,506.0,655
ANTI-SEBUM,0.5000,0.0192,0.0370,52.0,2
SKIN PROTECTING,0.3668,0.3971,0.3814,340.0,368
SKIN CONDITIONING - OCCLUSIVE,0.0000,0.0000,0.0000,2.0,0
SOOTHING,0.0000,0.0000,0.0000,21.0,0


### Linear SVM의 기능별 테스트 성능 확인


In [25]:
svm_test_function_table = function_metrics(y_test, svm_test_pred)
display(svm_test_function_table.round(4))


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.0625,0.0476,0.0541,21.0,16
SKIN CONDITIONING - EMOLLIENT,0.4852,0.5178,0.5010,506.0,540
ANTI-SEBUM,0.3333,0.1154,0.1714,52.0,18
SKIN PROTECTING,0.3897,0.3324,0.3587,340.0,290
SKIN CONDITIONING - OCCLUSIVE,0.5000,0.5000,0.5000,2.0,2
SOOTHING,0.4286,0.1429,0.2143,21.0,7


첫 번째 실험 결과: Logistic Regression은 Micro F1이 높았고, Linear SVM은 Macro F1이 높았다. Logistic Regression은 빈도가 높은 기능에서 상대적으로 우수했으며, SVM은 희귀 기능에서도 일부 정답을 찾아냈다. 다만 두 모델 모두 피부 보습·피지 조절·진정 기능의 예측 성능은 낮았다.

## 4. 클래스 불균형 보정과 초기 오류 분석

balanced 설정의 효과와 초기 모델의 추론 오류를 확인합니다. 이 절에서 저장한 cosmetic_function_models.joblib는 초기 비교용 파일이며 서비스의 최종 모델이 아닙니다.


### 클래스 불균형을 고려한 두 모델 재학습


In [26]:
# 클래스 불균형을 고려한 모델 생성
lr_balanced_model = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        solver="liblinear",
        max_iter=1000,
        class_weight="balanced",
        random_state=SEED
    ),
    n_jobs=1
)

svm_balanced_model = OneVsRestClassifier(
    LinearSVC(
        C=1.0,
        max_iter=5000,
        class_weight="balanced",
        random_state=SEED
    ),
    n_jobs=1
)

# Logistic Regression 학습
start_time = time.perf_counter()

lr_balanced_model.fit(X_train, y_train)

lr_balanced_training_time = time.perf_counter() - start_time

print("Balanced Logistic Regression 학습 완료!")
print(f"학습 시간: {lr_balanced_training_time:.2f}초")

# Linear SVM 학습
start_time = time.perf_counter()

svm_balanced_model.fit(X_train, y_train)

svm_balanced_training_time = time.perf_counter() - start_time

print("\nBalanced Linear SVM 학습 완료!")
print(f"학습 시간: {svm_balanced_training_time:.2f}초")

# 학습한 기능 수 확인
print("\nLogistic Regression 기능 수:",
      len(lr_balanced_model.estimators_))
print("Linear SVM 기능 수:",
      len(svm_balanced_model.estimators_))

Balanced Logistic Regression 학습 완료!
학습 시간: 7.51초

Balanced Linear SVM 학습 완료!
학습 시간: 6.19초

Logistic Regression 기능 수: 6
Linear SVM 기능 수: 6


### Balanced 모델의 기준값별 검증 성능 비교


In [27]:
# Balanced 모델의 검증 예측 점수
lr_balanced_val_proba = lr_balanced_model.predict_proba(X_val)
svm_balanced_val_scores = svm_balanced_model.decision_function(X_val)

# 모델별로 비교할 기준값
balanced_settings = {
    "Logistic Regression": {
        "scores": lr_balanced_val_proba,
        "thresholds": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    },
    "Linear SVM": {
        "scores": svm_balanced_val_scores,
        "thresholds": [-0.9, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 0.9]
    }
}

balanced_val_results = {}

for model_name, settings in balanced_settings.items():
    table, best_threshold = select_threshold(
        y_val, settings["scores"], settings["thresholds"])

    balanced_val_results[model_name] = {
        "table": table,
        "threshold": best_threshold,
        "prediction": (
            settings["scores"] >= best_threshold
        ).astype(int)
    }

    print(f"\nBalanced {model_name}")
    display(table.round(4))
    print("선택한 기준값:", best_threshold)

# 이후 단계에서 사용할 기준값 저장
lr_balanced_threshold = balanced_val_results[
    "Logistic Regression"
]["threshold"]

svm_balanced_threshold = balanced_val_results[
    "Linear SVM"
]["threshold"]


Balanced Logistic Regression


,기준값,Micro F1,Macro F1,Micro Precision,Micro Recall
0,0.1,0.2014,0.1362,0.1131,0.9170
1,0.2,0.2649,0.1780,0.1577,0.8268
2,0.3,0.3190,0.2190,0.2034,0.7387
3,0.4,0.3609,0.2451,0.2481,0.6619
4,0.5,0.3813,0.2656,0.2870,0.5676
5,0.6,0.4034,0.2948,0.3434,0.4887
6,0.7,0.3990,0.3022,0.4004,0.3975
7,0.8,0.3705,0.2987,0.4812,0.3012
8,0.9,0.2532,0.2604,0.5809,0.1619


선택한 기준값: 0.6

Balanced Linear SVM


,기준값,Micro F1,Macro F1,Micro Precision,Micro Recall
0,-0.90,0.2623,0.1640,0.1574,0.7879
1,-0.75,0.2938,0.1979,0.1837,0.7326
2,-0.50,0.3407,0.2480,0.2310,0.6486
3,-0.25,0.3744,0.2838,0.2796,0.5666
4,0.00,0.3940,0.2885,0.3329,0.4826
5,0.25,0.3905,0.2635,0.3773,0.4047
6,0.50,0.3736,0.2342,0.4558,0.3166
7,0.75,0.3143,0.1884,0.5358,0.2223
8,0.90,0.2724,0.1428,0.5884,0.1773


선택한 기준값: 0.0


### Balanced 모델의 기능별 검증 성능 확인


In [28]:
for model_name, result in balanced_val_results.items():
    function_table = function_metrics(y_val, result["prediction"])
    print(f"\nBalanced {model_name} — 기능별 검증 성능")
    display(function_table.round(4))



Balanced Logistic Regression — 기능별 검증 성능


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.0930,0.2162,0.1301,37.0,86
SKIN CONDITIONING - EMOLLIENT,0.4478,0.5455,0.4918,495.0,603
ANTI-SEBUM,0.1176,0.2941,0.1681,34.0,85
SKIN PROTECTING,0.3146,0.4674,0.3761,383.0,569
SKIN CONDITIONING - OCCLUSIVE,0.3750,0.3333,0.3529,9.0,8
SOOTHING,0.1842,0.3889,0.2500,18.0,38



Balanced Linear SVM — 기능별 검증 성능


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.1525,0.2432,0.1875,37.0,59
SKIN CONDITIONING - EMOLLIENT,0.3808,0.5515,0.4505,495.0,717
ANTI-SEBUM,0.1509,0.2353,0.1839,34.0,53
SKIN PROTECTING,0.3114,0.4569,0.3704,383.0,562
SKIN CONDITIONING - OCCLUSIVE,0.6667,0.2222,0.3333,9.0,3
SOOTHING,0.1905,0.2222,0.2051,18.0,21


### 기존·Balanced 모델의 검증 성능 한 번에 비교


In [29]:
validation_candidates = [
    {
        "모델": "Logistic Regression",
        "불균형 보정": "미적용",
        "기준값": lr_threshold,
        "예측": (
            lr_model.predict_proba(X_val) >= lr_threshold
        ).astype(int)
    },
    {
        "모델": "Linear SVM",
        "불균형 보정": "미적용",
        "기준값": svm_threshold,
        "예측": (
            svm_model.decision_function(X_val) >= svm_threshold
        ).astype(int)
    },
    {
        "모델": "Logistic Regression",
        "불균형 보정": "balanced",
        "기준값": lr_balanced_threshold,
        "예측": balanced_val_results[
            "Logistic Regression"
        ]["prediction"]
    },
    {
        "모델": "Linear SVM",
        "불균형 보정": "balanced",
        "기준값": svm_balanced_threshold,
        "예측": balanced_val_results[
            "Linear SVM"
        ]["prediction"]
    }
]

comparison_rows = []

for candidate in validation_candidates:
    comparison_rows.append({
        "모델": candidate["모델"],
        "불균형 보정": candidate["불균형 보정"],
        "기준값": candidate["기준값"],
        **evaluate_predictions(y_val, candidate["예측"])
    })

validation_comparison = pd.DataFrame(comparison_rows)

display(validation_comparison.round(4))

,모델,불균형 보정,기준값,Micro F1,Macro F1,Micro Precision,Micro Recall
0,Logistic Regression,미적용,0.20,0.4301,0.1806,0.4239,0.4365
1,Linear SVM,미적용,-0.25,0.4028,0.2278,0.4220,0.3852
2,Logistic Regression,balanced,0.60,0.4034,0.2948,0.3434,0.4887
3,Linear SVM,balanced,0.00,0.3940,0.2885,0.3329,0.4826


### 모델·변환기·기준값 저장 및 다운로드


In [30]:
import joblib
import sklearn
from google.colab import files

model_bundle = {
    # 입력 변환과 출력 기능 순서
    "vectorizer": vectorizer,
    "labels": list(mlb.classes_),

    # 검증 Micro F1 기준 선택
    "selected_model": "lr_default",
    "selection_metric": "validation_micro_f1",

    # 모델별 예측 방법과 기준값
    "models": {
        "lr_default": {
            "model": lr_model,
            "score_method": "predict_proba",
            "threshold": float(lr_threshold)
        },
        "svm_default": {
            "model": svm_model,
            "score_method": "decision_function",
            "threshold": float(svm_threshold)
        },
        "lr_balanced": {
            "model": lr_balanced_model,
            "score_method": "predict_proba",
            "threshold": float(lr_balanced_threshold)
        },
        "svm_balanced": {
            "model": svm_balanced_model,
            "score_method": "decision_function",
            "threshold": float(svm_balanced_threshold)
        }
    },

    # 재현과 결과 기록
    "seed": SEED,
    "sklearn_version": sklearn.__version__,
    "validation_results": validation_comparison.to_dict(
        orient="records"
    )
}

save_path = "cosmetic_function_models.joblib"

joblib.dump(model_bundle, save_path, compress=3)

print("저장 완료:", save_path)
print("선택 모델:", model_bundle["selected_model"])
print("기능 수:", len(model_bundle["labels"]))

# 내 컴퓨터로 다운로드
files.download(save_path)

저장 완료: cosmetic_function_models.joblib
선택 모델: lr_default
기능 수: 6


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

두번째 실험 결론: 불균형 보정은 희귀 기능의 탐지에 도움이 됐지만, 오탐 증가와 전체 F1 감소가 동반됐다

### 저장한 초기 모델로 성분명 예측


In [31]:
import joblib
import pandas as pd
import unicodedata
import re

# 저장한 파일 불러오기
loaded_bundle = joblib.load("cosmetic_function_models.joblib")

# 선택된 모델과 필요한 정보 가져오기
model_name = loaded_bundle["selected_model"]
model_info = loaded_bundle["models"][model_name]

loaded_model = model_info["model"]
loaded_vectorizer = loaded_bundle["vectorizer"]
loaded_labels = loaded_bundle["labels"]
loaded_threshold = model_info["threshold"]

print("불러온 모델:", model_name)
print("예측 기준값:", loaded_threshold)
print("예측할 기능:", loaded_labels)

불러온 모델: lr_default
예측 기준값: 0.2
예측할 기능: ['SKIN CONDITIONING - HUMECTANT', 'SKIN CONDITIONING - EMOLLIENT', 'ANTI-SEBUM', 'SKIN PROTECTING', 'SKIN CONDITIONING - OCCLUSIVE', 'SOOTHING']


### 성분 하나를 입력해 기능 예측


In [33]:
function_names = {
    "SKIN CONDITIONING - HUMECTANT": "피부 보습",
    "SKIN CONDITIONING - EMOLLIENT": "피부 유연화",
    "ANTI-SEBUM": "피지 조절",
    "SKIN PROTECTING": "피부 보호",
    "SKIN CONDITIONING - OCCLUSIVE": "수분 증발 억제",
    "SOOTHING": "피부·두피 불편감 완화"
}

def predict_ingredient(ingredient_name):
    # 학습 때와 같은 성분명 전처리
    name = unicodedata.normalize("NFKC", ingredient_name)
    name = re.sub(r"\s+", " ", name).strip().upper()

    if not name:
        raise ValueError("성분명을 입력해 주세요.")

    # 성분명을 숫자 특징으로 변환
    X_input = loaded_vectorizer.transform([name])

    # 저장된 모델에 맞는 예측 방법 사용
    score_method = getattr(
        loaded_model,
        model_info["score_method"]
    )
    scores = score_method(X_input)[0]

    # 기준값을 넘으면 1로 판단
    predictions = scores >= loaded_threshold

    result = pd.DataFrame({
        "기능": loaded_labels,
        "설명": [
            function_names.get(label, label)
            for label in loaded_labels
        ],
        "모델 점수": scores,
        "예측": predictions.astype(int)
    })

    print("입력 성분:", name)

    if X_input.nnz == 0:
        print("주의: 학습한 단어·글자 패턴과 겹치는 특징이 없습니다.")

    return result

# 따옴표 안의 성분명을 바꿔서 실행
result = predict_ingredient("GLYCERIN")
display(result.round(4))

입력 성분: GLYCERIN


,기능,설명,모델 점수,예측
0,SKIN CONDITIONING - HUMECTANT,피부 보습,0.0038,0
1,SKIN CONDITIONING - EMOLLIENT,피부 유연화,0.4450,1
2,ANTI-SEBUM,피지 조절,0.0037,0
3,SKIN PROTECTING,피부 보호,0.1966,0
4,SKIN CONDITIONING - OCCLUSIVE,수분 증발 억제,0.0034,0
5,SOOTHING,피부·두피 불편감 완화,0.0030,0


GLYCERIN 입력 시 피부 유연화를 양성으로 예측했으나 원본에는 해당 라벨이 없었으며, 원본에 기록된 피부 보호 기능은 음성으로 예측하였다. 또한 일반적인 보습 개념과 데이터셋의 세부 라벨 구분에 차이가 있어 결과 해석에 주의가 필요했다.

### 여러 성분을 한 번에 입력해 기능 예측


In [34]:
# 분석할 성분 입력: 한 줄에 하나씩
ingredient_text = """
GLYCERIN
NIACINAMIDE
SALICYLIC ACID
"""

# 빈 줄 제거 및 중복 정리
ingredient_names = list(dict.fromkeys(
    line.strip()
    for line in ingredient_text.splitlines()
    if line.strip()
))

if not ingredient_names:
    raise ValueError("성분명을 한 개 이상 입력해 주세요.")

# 성분별 예측 결과 모으기
prediction_tables = []

for ingredient in ingredient_names:
    result = predict_ingredient(ingredient)

    result.insert(
        0,
        "성분명",
        unicodedata.normalize("NFKC", ingredient)
        .upper()
        .strip()
    )

    prediction_tables.append(result)

all_predictions = pd.concat(
    prediction_tables,
    ignore_index=True
)

display(all_predictions.round(4))

입력 성분: GLYCERIN
입력 성분: NIACINAMIDE
입력 성분: SALICYLIC ACID


,성분명,기능,설명,모델 점수,예측
0,GLYCERIN,SKIN CONDITIONING - HUMECTANT,피부 보습,0.0038,0
1,GLYCERIN,SKIN CONDITIONING - EMOLLIENT,피부 유연화,0.4450,1
2,GLYCERIN,ANTI-SEBUM,피지 조절,0.0037,0
3,GLYCERIN,SKIN PROTECTING,피부 보호,0.1966,0
4,GLYCERIN,SKIN CONDITIONING - OCCLUSIVE,수분 증발 억제,0.0034,0
5,GLYCERIN,SOOTHING,피부·두피 불편감 완화,0.0030,0
6,NIACINAMIDE,SKIN CONDITIONING - HUMECTANT,피부 보습,0.0061,0
7,NIACINAMIDE,SKIN CONDITIONING - EMOLLIENT,피부 유연화,0.0254,0
8,NIACINAMIDE,ANTI-SEBUM,피지 조절,0.0148,0
9,NIACINAMIDE,SKIN PROTECTING,피부 보호,0.1106,0


### 원본 등록 기능과 모델 예측 비교


In [35]:
# 학습 때와 동일하게 성분명 정리
def normalize_name(name):
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip().upper()

# 성분명 → 원본에 기록된 기능 목록
function_lookup = {
    normalize_name(name): set(labels)
    for name, labels in zip(data["INCI name"], data["labels"])
}

# 예측표의 성분명도 동일하게 정리
all_predictions["성분명"] = (
    all_predictions["성분명"].apply(normalize_name)
)

# 원본 등록 여부와 모델 예측을 비교
def get_registered_status(row):
    registered = function_lookup.get(row["성분명"])

    if registered is None:
        return "조회되지 않음"

    if row["기능"] in registered:
        return "기록됨"

    return "미기록"

comparison_table = all_predictions.copy()

comparison_table["원본 등록 여부"] = comparison_table.apply(
    get_registered_status,
    axis=1
)

comparison_table = comparison_table.rename(
    columns={"예측": "모델 예측"}
)

display(
    comparison_table[
        [
            "성분명",
            "설명",
            "원본 등록 여부",
            "모델 점수",
            "모델 예측"
        ]
    ].round(4)
)

,성분명,설명,원본 등록 여부,모델 점수,모델 예측
0,GLYCERIN,피부 보습,미기록,0.0038,0
1,GLYCERIN,피부 유연화,미기록,0.4450,1
2,GLYCERIN,피지 조절,미기록,0.0037,0
3,GLYCERIN,피부 보호,기록됨,0.1966,0
4,GLYCERIN,수분 증발 억제,미기록,0.0034,0
5,GLYCERIN,피부·두피 불편감 완화,미기록,0.0030,0
6,NIACINAMIDE,피부 보습,미기록,0.0061,0
7,NIACINAMIDE,피부 유연화,미기록,0.0254,0
8,NIACINAMIDE,피지 조절,미기록,0.0148,0
9,NIACINAMIDE,피부 보호,미기록,0.1106,0


### 입력 성분들의 기능 요약


In [36]:
summary_rows = []

for label in loaded_labels:
    # 원본에 해당 기능이 기록된 입력 성분
    registered_names = [
        name
        for name in dict.fromkeys(
            normalize_name(n) for n in ingredient_names
        )
        if label in function_lookup.get(name, set())
    ]

    # 모델이 해당 기능을 양성으로 예측한 입력 성분
    predicted_names = (
        all_predictions.loc[
            (all_predictions["기능"] == label)
            & (all_predictions["예측"] == 1),
            "성분명"
        ]
        .drop_duplicates()
        .tolist()
    )

    summary_rows.append({
        "기능": function_names.get(label, label),
        "원본 등록 성분": ", ".join(registered_names) or "없음",
        "모델 예측 성분": ", ".join(predicted_names) or "없음"
    })

ingredient_summary = pd.DataFrame(summary_rows)

display(ingredient_summary)

,기능,원본 등록 성분,모델 예측 성분
0,피부 보습,없음,없음
1,피부 유연화,없음,GLYCERIN
2,피지 조절,없음,없음
3,피부 보호,GLYCERIN,없음
4,수분 증발 억제,없음,없음
5,피부·두피 불편감 완화,없음,없음


성분명만으로 CosIng의 세부 기능을 예측하는 데 정보적 한계가 있으며, 희귀 라벨의 사례 부족과 클래스 불균형이 이를 악화시킨 것으로 보인다.

## 5. 입력 개선: 성분명 + 설명

기존 데이터 분할을 유지하면서 화학적 설명을 추가합니다. TF-IDF 변환기는 학습 데이터에서만 학습합니다.


### 세 번째 문제 · 성분명만으로 세부 기능을 판단하기 어려움

**세 번째 문제:** GLYCERIN 등 입력에서 등록 기능을 놓치거나 다른 기능을 예측하는 오류를 확인했다.

**세 번째 해결:** 성분명에 CosIng의 화학적 설명을 붙여 TF-IDF 입력을 확장했다.

**세 번째 한계 — 실험 후 확인:** 검증 성능은 개선됐지만 설명은 효능 실험 데이터가 아니며, 짧거나 없는 설명과 세부 라벨의 불균형이 남았다.


### 성분 설명에 추가 정보가 있는지 확인


In [38]:
description_column = "Chem/IUPAC Name / Description"

# 설명 확인용 데이터 복사
description_data = df[
    ["INCI name", description_column, "Function"]
].copy()

# 기존과 같은 방식으로 성분명 정리
description_data["INCI name"] = (
    description_data["INCI name"].apply(normalize_name)
)

# 설명이 없는 경우 빈 문자열로 처리
description_data[description_column] = (
    description_data[description_column]
    .fillna("")
    .astype(str)
    .str.strip()
)

# 학습 데이터에 속한 성분만 살펴보기
train_description_data = description_data[
    description_data["INCI name"].isin(
        train_data["INCI name"]
    )
].copy()

print("설명이 있는 학습 행 수:",
      train_description_data[description_column].ne("").sum())

print("설명이 없는 학습 행 수:",
      train_description_data[description_column].eq("").sum())

train_description_data["설명 길이"] = (
    train_description_data[description_column].str.len()
)

nonempty_descriptions = train_description_data[
    train_description_data["설명 길이"] > 0
]

columns_to_show = [
    "INCI name",
    description_column,
    "Function",
    "설명 길이"
]

# 긴 설명도 볼 수 있도록 표시 길이 설정
with pd.option_context("display.max_colwidth", 250):
    print("설명이 짧은 사례")
    display(
        nonempty_descriptions
        .sort_values(["설명 길이", "INCI name"])
        [columns_to_show]
        .head(5)
    )

    print("\n설명이 긴 사례")
    display(
        nonempty_descriptions
        .sort_values(
            ["설명 길이", "INCI name"],
            ascending=[False, True]
        )
        [columns_to_show]
        .head(5)
    )

설명이 있는 학습 행 수: 20199
설명이 없는 학습 행 수: 316
설명이 짧은 사례


,INCI name,Chem/IUPAC Name / Description,Function,설명 길이
10760,GALACTOOLIGOSACCHARIDES,GOS,"HUMECTANT, SKIN CONDITIONING",3
20855,PHYTOHEMAGGLUTININ,PHA,SKIN CONDITIONING - MISCELLANEOUS,3
24616,S-TERTIAPIN SP,TPN,SKIN PROTECTING,3
25627,SH-POLYPEPTIDE-13,NOG,SKIN PROTECTING,3
27197,SR-SILKWORM POLYPEPTIDE-2,SP1,SKIN CONDITIONING,3



설명이 긴 사례


,INCI name,Chem/IUPAC Name / Description,Function,설명 길이
14465,LACTOBACILLUS/CINNAMOMUM ZEYLANICUM BARK/(ARALIA CORDATA/SOLANUM MELONGENA/CAPSICUM ANNUUM/CUCUMIS MELO/ABELMOSCHUS ESCULENTUS/CUCURBITA MAXIMA/CUCURBITA MOSCHATA/CUCUMIS SATIVUS/CITRULLUS LANATUS/BENINCASA HISPIDA/MORMORIDCA CHARANTIA/CITRUS AUR...,"This is a filtrate of the product obtained by the fermentation of sugar and the bark of Cinnamomum zeylanicum ; the fruit of Aralia cordata, Solanum melongena, Capsicum annuum, Cucumis melo, Abelmoschus esculentus, Cucurbita maxima, Cucurbita mos...",SKIN CONDITIONING - MISCELLANEOUS,2233
2837,ASPERGILLUS/TURNIP ROOT/EUTERPE OLERACEA FRUIT/PHASEOLUS ANGULARIS SEED/ACEROLA FRUIT/ANISE FRUIT/AVOCADO FRUIT/GYNOSTEMMA PENTAPHYLLUM LEAF/PANICUM ITALICUM SEED/CICER ARIETINUM SEED/PEA SEED/OAT SEED/HORDEUM VULGARE SEED/ORANGE FRUIT/EQUISETUM ...,"This is the extract of the product obtained by the fermentation of the roots of Brassica rapa (turnip), Curcuma zedoaria, Arctium lappa, Zingiber officinale (ginger), Solanum tuberosum (potato), Daucus carota (carrot), Pfaffia paniculata, Lepidiu...",SKIN CONDITIONING,2178
22,(ADONIS AMURENSIS/AKEBIA QUINATA/ALBIZIA JULIBRISSIN/ALTHAEA ROSEA/ANGELICA ACUTILOBA/ANGELICA ANOMALA/AQUILEGIA BUERGERIANA/ARTEMISIA PRINCEPS/ASTER CILIOSUS/ASTER SCABER/BRASSICA CAMPESTRIS/BROUSSONETIA KAZINOKI/CALLISTEPHUS CHINENSIS/CAMELLIA ...,"This is the extract of the flowers of Adonis amurensis, Akebia quinata, Albizia julibrissin, Althaea rosea, Angelica acutiloba, Angelica anomala, Aquilegia buergeriana, Artemisia princeps, Aster ciliosus, Aster scaber, Brassica campestris, Brouss...","SKIN CONDITIONING - HUMECTANT, SKIN PROTECTING",2000
862,ACETOBACTER/(ACANTHOPANAX SESSILIFLORUS/AILANTHUS ALTISSIMA/ALBIZIA JULIBRISSIN/BETULA ALBA/BETULA PLATYPHYLLA JAPONICA/EUCOMMIA ULMOIDES/MELIA AZADIRACHTA/MORUS ALBA) BARK/(ADENOPHORA STRICTA/ANGELICA ACUTILOBA/ANGELICA KOREANA/ANGELICA TENUISSI...,"This is the filtrate of the product obtained by the fermentation of the bark of Acanthopanax sessiliflorus, Ailanthus altissima, Albizia julibrissin, Betula alba, Betula platyphylla japonica, Eucommia ulmoides, Melia azadirachta, Morus alba ; the...",SKIN CONDITIONING - MISCELLANEOUS,2000
1390,ADONIS AMURENSIS/AKEBIA QUINATA/ALBIZIA JULIBRISSIN/ALTHAEA ROSEA/ANGELICA ACUTILOBA/ANGELICA ANOMALA/AQUILEGIA BUERGERIANA/ARTEMISIA PRINCEPS/ASTER CILIOSUS/ASTER SCABER/BRASSICA CAMPESTRIS/BROUSSONETIA KAZINOKI/CALLISTEPHUS CHINENSIS/CAMELLIA J...,This is the extract of the flowers of Adonis Amurensis/Akebia Quinata/Albizia Julibrissin/Althaea Rosea/Angelica Acutiloba/Angelica Anomala/Aquilegia Buergeriana/Artemisia Princeps/Aster Ciliosus/Aster Scaber/Brassica Campestris/Broussonetia Kazi...,SKIN CONDITIONING,1890


### 같은 성분의 설명을 통합하고 입력 문장 만들기


In [39]:
# 성분별 중복 설명을 제거하고 통합
description_lookup = (
    description_data
    .groupby("INCI name")[description_column]
    .agg(
        lambda values: " ".join(
            dict.fromkeys(
                text for text in values if text
            )
        )
    )
    .to_dict()
)

# 성분명 + 설명으로 입력 생성
def make_description_input(split_data):
    names = split_data["INCI name"]

    descriptions = names.map(
        description_lookup
    ).fillna("")

    return names + " " + descriptions

X_train_desc_text = make_description_input(train_data)
X_val_desc_text = make_description_input(val_data)

# 입력과 기존 정답의 행 수가 같은지 확인
assert len(X_train_desc_text) == len(y_train)
assert len(X_val_desc_text) == len(y_val)

# 학습 데이터의 입력 예시 확인
with pd.option_context("display.max_colwidth", 300):
    display(
        pd.DataFrame({
            "성분명": train_data["INCI name"].head(5),
            "새 입력": X_train_desc_text.head(5)
        })
    )

,성분명,새 입력
0,NERYL ISOVALERATEATE,"NERYL ISOVALERATEATE cis-3,7-Dimethyl-2,6-octadienyl 3-methyl-butanoate"
1,GRACILARIA CONFERTA POWDER,"GRACILARIA CONFERTA POWDER Gracilaria Conferta Powder is the powder obtained from the dried, ground alga, Gracilaria conferta, Gracilariaceae."
2,LILIUM SPECIOSUM RUBRUM BULB EXTRACT,"LILIUM SPECIOSUM RUBRUM BULB EXTRACT Lilium Speciosum Rubrum Bulb Extract is an extract of the bulb of Lilium Speciosum rubrum, Liliaceae"
3,ROSMARINUS OFFICINALIS LEAF/STEM OIL,"ROSMARINUS OFFICINALIS LEAF/STEM OIL Rosmarinus Officinalis (Rosemary) Leaf/Stem Oil is the volatile oil obtained from the leaves and stems of Rosmarinus officinalis, Lamiaceae."
4,GAULTHERIA PROCUMBENS LEAF EXTRACT,"GAULTHERIA PROCUMBENS LEAF EXTRACT Gaultheria Procumbens Leaf Extract is an extract of the leaves of the Wintergreen, Gaultheria procumbens L., Ericaceae"


### 성분명+설명을 TF-IDF로 변환


In [40]:
from sklearn.base import clone

# 설정만 복사하고 학습 상태는 복사하지 않음
desc_vectorizer = clone(vectorizer)

# 학습 데이터에서만 TF-IDF 학습
X_train_desc = desc_vectorizer.fit_transform(
    X_train_desc_text
)

X_val_desc = desc_vectorizer.transform(
    X_val_desc_text
)

print("학습 입력 크기:", X_train_desc.shape)
print("검증 입력 크기:", X_val_desc.shape)

학습 입력 크기: (20514, 80000)
검증 입력 크기: (4396, 80000)


### 성분명+설명으로 모델 4개 학습


In [41]:
desc_models, desc_training_times = fit_variants(X_train_desc, y_train)


LR 기본 학습 완료: 10.54초
SVM 기본 학습 완료: 4.94초
LR balanced 학습 완료: 18.88초
SVM balanced 학습 완료: 13.01초


### 검증 기준값 선택 및 기존 입력과 비교


In [42]:
# 기존 실험과 동일한 기준값 후보
desc_settings = {
    "LR 기본": (
        "predict_proba",
        [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]
    ),
    "SVM 기본": (
        "decision_function",
        [-0.9, -0.75, -0.5, -0.25, 0.0, 0.25]
    ),
    "LR balanced": (
        "predict_proba",
        [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    ),
    "SVM balanced": (
        "decision_function",
        [-0.9, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 0.9]
    )
}

desc_val_results, desc_validation_comparison = validate_variants(
    desc_models, X_val_desc, y_val, desc_settings, "입력", "성분명+설명")

# 이전 성분명 전용 실험 결과 정리
name_only_comparison = validation_comparison.copy()

name_only_comparison["설정"] = (
    name_only_comparison["모델"].map({
        "Logistic Regression": "LR",
        "Linear SVM": "SVM"
    })
    + " "
    + name_only_comparison["불균형 보정"].map({
        "미적용": "기본",
        "balanced": "balanced"
    })
)

name_only_comparison["입력"] = "성분명"

columns = [
    "설정", "입력", "기준값",
    "Micro F1", "Macro F1",
    "Micro Precision", "Micro Recall"
]

input_comparison = pd.concat(
    [
        name_only_comparison[columns],
        desc_validation_comparison[columns]
    ],
    ignore_index=True
).sort_values(["설정", "입력"])

display(input_comparison.round(4))

,설정,입력,기준값,Micro F1,Macro F1,Micro Precision,Micro Recall
2,LR balanced,성분명,0.60,0.4034,0.2948,0.3434,0.4887
6,LR balanced,성분명+설명,0.60,0.4395,0.3259,0.3831,0.5154
0,LR 기본,성분명,0.20,0.4301,0.1806,0.4239,0.4365
4,LR 기본,성분명+설명,0.20,0.4425,0.1923,0.4336,0.4518
3,SVM balanced,성분명,0.00,0.3940,0.2885,0.3329,0.4826
7,SVM balanced,성분명+설명,0.00,0.4367,0.3490,0.3967,0.4857
1,SVM 기본,성분명,-0.25,0.4028,0.2278,0.4220,0.3852
5,SVM 기본,성분명+설명,-0.25,0.4461,0.3592,0.4756,0.4201


### 최종 후보 고정 및 테스트 평가


In [44]:

# 검증 Micro F1이 가장 높은 설정 선택
best_index = desc_validation_comparison["Micro F1"].idxmax()
final_desc_name = desc_validation_comparison.loc[
    best_index, "설정"
]

final_desc_model = desc_models[final_desc_name]
final_desc_result = desc_val_results

# 검증 결과로 최종 설정 고정
final_model_name = "SVM 기본"
final_model = desc_models[final_model_name]

final_threshold = desc_val_results[
    final_model_name
]["threshold"]

final_score_method = desc_val_results[
    final_model_name
]["score_method"]

# 테스트 입력 생성: 성분명 + 설명
X_test_desc_text = make_description_input(test_data)

# 학습 데이터에 맞춘 변환기를 그대로 적용
X_test_desc = desc_vectorizer.transform(
    X_test_desc_text
)

# 테스트 예측
final_test_scores = getattr(
    final_model,
    final_score_method
)(X_test_desc)

final_test_pred = (
    final_test_scores >= final_threshold
).astype(int)

print("선택 모델:", final_model_name)
print("입력: 성분명 + 설명")
print("기준값:", final_threshold)

# 전체 성능
final_test_metrics = evaluate_predictions(
    y_test,
    final_test_pred
)

display(
    pd.DataFrame(
        final_test_metrics.items(),
        columns=["평가지표", "점수"]
    ).round(4)
)

# 기능별 성능
final_desc_report = classification_report(
    y_test,
    final_test_pred,
    target_names=mlb.classes_,
    output_dict=True,
    zero_division=0
)

final_desc_function_table = (
    pd.DataFrame(final_desc_report)
    .T
    .loc[
        list(mlb.classes_),
        ["precision", "recall", "f1-score", "support"]
    ]
    .copy()
)

final_desc_function_table["양성 예측 수"] = (
    final_test_pred.sum(axis=0)
)

display(final_desc_function_table.round(4))

선택 모델: SVM 기본
입력: 성분명 + 설명
기준값: -0.25


,평가지표,점수
0,Micro F1,0.4698
1,Macro F1,0.3604
2,Micro Precision,0.4826
3,Micro Recall,0.4575


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.0909,0.0476,0.0625,21.0,11
SKIN CONDITIONING - EMOLLIENT,0.5347,0.5632,0.5486,506.0,533
ANTI-SEBUM,0.4667,0.1346,0.2090,52.0,15
SKIN PROTECTING,0.4092,0.3912,0.4000,340.0,325
SKIN CONDITIONING - OCCLUSIVE,1.0000,0.5000,0.6667,2.0,1
SOOTHING,0.5000,0.1905,0.2759,21.0,8


성분명에 화학적 설명을 추가하면 기능 분류 성능이 개선되었다. 그러나 희귀 기능의 낮은 재현율과 적은 평가 사례 수로 인해, 6개 기능 모두를 안정적으로 예측하는 데에는 한계가 있었다. 따라서 단어·글자 패턴을 기반으로 하는 TF-IDF와 사전 학습된 문장 표현을 사용하는 MiniLM을 비교하여, 텍스트 표현 방식의 변경이 기능 분류 성능을 개선하는지 확인한다. MiniLM은 고정된 특징 추출기로 사용하고, 추출된 임베딩으로 Logistic Regression과 Linear SVM을 학습한다. 기존 데이터 분할을 유지하며 모델 설정은 검증 데이터로 선택한다.

## 6. 표현 방식 비교: MiniLM

사전 학습 MiniLM은 고정 특징 추출기로 사용하며, 분류기를 별도로 학습합니다. MiniLM 자체의 파인튜닝 실험은 아닙니다.


### 네 번째 문제 · 단어 패턴을 넘어선 표현이 필요할 수 있음

**네 번째 문제:** 성분명+설명 TF-IDF에서도 희귀 기능의 재현율이 낮았다.

**네 번째 해결:** 고정 MiniLM 임베딩을 추출하고 동일한 네 분류기 설정으로 TF-IDF와 비교했다.

**네 번째 한계 — 실험 후 확인:** 이번 검증에서는 TF-IDF보다 낮은 성능을 보였다. 일반 문장 임베딩이 화학 기능 분류에 맞지 않을 가능성이 있고 최대 256토큰에서 긴 설명이 잘린다. 원인을 분리해 검증한 실험은 아니다.


### MiniLM 사용을 위한 라이브러리 설치


In [45]:
%pip install -q sentence-transformers

### MiniLM 불러오기 및 성분명+설명 변환


In [46]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer

# GPU가 있으면 사용하고, 없으면 CPU 사용
device = "cuda" if torch.cuda.is_available() else "cpu"
print("실행 장치:", device)

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    embedding_model_name,
    device=device
)

print("최대 입력 토큰 수:", embedding_model.max_seq_length)

# 학습 입력 변환
X_train_minilm = embedding_model.encode(
    X_train_desc_text.tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# 검증 입력 변환
X_val_minilm = embedding_model.encode(
    X_val_desc_text.tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("학습 입력 크기:", X_train_minilm.shape)
print("검증 입력 크기:", X_val_minilm.shape)

# 정답과 입력의 행 수 확인
assert len(X_train_minilm) == len(y_train)
assert len(X_val_minilm) == len(y_val)

실행 장치: cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (8 outputs left out)

최대 입력 토큰 수: 256


Batches:   0%|          | 0/321 [00:00<?, ?it/s]

Batches:   0%|          | 0/69 [00:00<?, ?it/s]

학습 입력 크기: (20514, 384)
검증 입력 크기: (4396, 384)


### 변환 결과 저장


In [47]:
import numpy as np
from google.colab import files

np.savez_compressed(
    "minilm_train_val_embeddings.npz",
    X_train=X_train_minilm,
    X_val=X_val_minilm,
    y_train=y_train,
    y_val=y_val,
    train_names=train_data["INCI name"].to_numpy(dtype=str),
    val_names=val_data["INCI name"].to_numpy(dtype=str),
    labels=np.array(mlb.classes_, dtype=str),
    model_name=np.array(embedding_model_name),
    max_seq_length=np.array(embedding_model.max_seq_length),
    normalize_embeddings=np.array(True)
)

files.download("minilm_train_val_embeddings.npz")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### MiniLM 임베딩으로 분류기 4개 학습


In [48]:
minilm_models, minilm_training_times = fit_variants(X_train_minilm, y_train)


LR 기본 학습 완료: 14.33초
SVM 기본 학습 완료: 22.09초
LR balanced 학습 완료: 27.85초
SVM balanced 학습 완료: 33.03초


### MiniLM 모델의 검증 성능 확인 및 TF-IDF와 비교


In [49]:
minilm_val_results, minilm_validation_comparison = validate_variants(
    minilm_models, X_val_minilm, y_val, desc_settings, "텍스트 표현", "MiniLM")

# 동일한 성분명+설명을 사용한 TF-IDF 실험과 비교
tfidf_comparison = desc_validation_comparison.copy()
tfidf_comparison["텍스트 표현"] = "TF-IDF"

comparison_columns = [
    "설정", "텍스트 표현", "기준값",
    "Micro F1", "Macro F1",
    "Micro Precision", "Micro Recall"
]

representation_comparison = pd.concat(
    [
        tfidf_comparison[comparison_columns],
        minilm_validation_comparison[comparison_columns]
    ],
    ignore_index=True
).sort_values(["설정", "텍스트 표현"])

display(representation_comparison.round(4))

,설정,텍스트 표현,기준값,Micro F1,Macro F1,Micro Precision,Micro Recall
6,LR balanced,MiniLM,0.70,0.2645,0.2046,0.1958,0.4078
2,LR balanced,TF-IDF,0.60,0.4395,0.3259,0.3831,0.5154
4,LR 기본,MiniLM,0.20,0.3581,0.1226,0.3471,0.3699
0,LR 기본,TF-IDF,0.20,0.4425,0.1923,0.4336,0.4518
7,SVM balanced,MiniLM,0.25,0.2737,0.2380,0.1902,0.4877
3,SVM balanced,TF-IDF,0.00,0.4367,0.3490,0.3967,0.4857
5,SVM 기본,MiniLM,-0.50,0.3710,0.2036,0.3842,0.3586
1,SVM 기본,TF-IDF,-0.25,0.4461,0.3592,0.4756,0.4201


### MiniLM 모델의 기능별 검증 성능 확인


In [50]:
for name, result in minilm_val_results.items():
    function_table = function_metrics(y_val, result["prediction"])
    print(f"\nMiniLM + {name} — 기능별 검증 성능")
    display(function_table.round(4))



MiniLM + LR 기본 — 기능별 검증 성능


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.0000,0.0000,0.0000,37.0,0
SKIN CONDITIONING - EMOLLIENT,0.3636,0.4768,0.4126,495.0,649
ANTI-SEBUM,0.0000,0.0000,0.0000,34.0,0
SKIN PROTECTING,0.3197,0.3264,0.3230,383.0,391
SKIN CONDITIONING - OCCLUSIVE,0.0000,0.0000,0.0000,9.0,0
SOOTHING,0.0000,0.0000,0.0000,18.0,0



MiniLM + SVM 기본 — 기능별 검증 성능


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.0000,0.0000,0.0000,37.0,0
SKIN CONDITIONING - EMOLLIENT,0.3863,0.4667,0.4227,495.0,598
ANTI-SEBUM,0.0000,0.0000,0.0000,34.0,0
SKIN PROTECTING,0.3754,0.3029,0.3353,383.0,309
SKIN CONDITIONING - OCCLUSIVE,1.0000,0.2222,0.3636,9.0,2
SOOTHING,0.5000,0.0556,0.1000,18.0,2



MiniLM + LR balanced — 기능별 검증 성능


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.0402,0.3784,0.0727,37.0,348
SKIN CONDITIONING - EMOLLIENT,0.4012,0.4061,0.4036,495.0,501
ANTI-SEBUM,0.0303,0.3824,0.0562,34.0,429
SKIN PROTECTING,0.2759,0.4178,0.3323,383.0,580
SKIN CONDITIONING - OCCLUSIVE,0.2500,0.3333,0.2857,9.0,12
SOOTHING,0.0429,0.3889,0.0773,18.0,163



MiniLM + SVM balanced — 기능별 검증 성능


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.0303,0.2973,0.0550,37.0,363
SKIN CONDITIONING - EMOLLIENT,0.3503,0.5010,0.4123,495.0,708
ANTI-SEBUM,0.0327,0.4706,0.0612,34.0,489
SKIN PROTECTING,0.2306,0.5039,0.3164,383.0,837
SKIN CONDITIONING - OCCLUSIVE,1.0000,0.3333,0.5000,9.0,3
SOOTHING,0.0490,0.2778,0.0833,18.0,102


사전 학습된 MiniLM을 고정 특징 추출기로 사용하여 TF-IDF와 비교하였다. 동일한 데이터 분할에서 네 가지 분류기 설정 모두 TF-IDF보다 낮은 검증 Micro F1과 Macro F1을 보였다. 따라서 이번 실험에서는 성분명과 화학적 설명의 TF-IDF 표현을 사용하는 Linear SVM을 최종 후보로 유지하였다. 사전 학습된 문장 표현이 해당 도메인의 기능 분류 성능 향상을 보장하지 않음을 확인하였다.

## 7. SVM 기준값 및 규제 강도 탐색

기능별 기준값과 C 값을 검증셋에서 비교합니다. 큰 C에서 나타나는 학습·검증 성능 격차도 확인합니다.


### 다섯 번째 문제 · 기능마다 적합한 결정 기준이 다를 수 있음

**다섯 번째 문제:** 하나의 공통 기준값이 모든 기능에서 같은 정밀도·재현율 균형을 만들지는 못했다.

**다섯 번째 해결:** 검증셋에서 기능별 F1을 기준으로 개별 기준값을 탐색했다.

**다섯 번째 한계 — 실험 후 확인:** Macro F1 개선은 작았고 Micro F1은 소폭 하락했다. 희귀 기능의 검증 사례가 적어 기준값이 검증셋에 과도하게 맞춰질 수 있다.


### TF-IDF SVM의 기능별 기준값 조정


In [51]:
import numpy as np

# 성분명+설명으로 학습한 TF-IDF SVM
threshold_model = desc_models["SVM 기본"]

# 검증 데이터의 기능별 결정 점수
svm_desc_val_scores = threshold_model.decision_function(
    X_val_desc
)

# 모든 기능에 동일한 후보 범위 적용
threshold_candidates = np.array([
    -0.9, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 0.9
])

# 기존 공통 기준값
common_threshold = desc_val_results[
    "SVM 기본"
]["threshold"]

per_function_thresholds = []
threshold_summary_rows = []

for index, label in enumerate(mlb.classes_):
    y_true = y_val[:, index]
    scores = svm_desc_val_scores[:, index]

    # 기능별로 후보 기준값 평가
    candidate_f1 = [
        f1_score(
            y_true,
            (scores >= threshold).astype(int),
            zero_division=0
        )
        for threshold in threshold_candidates
    ]

    best_f1 = max(candidate_f1)

    # 동점이면 기존 공통 기준값에 가장 가까운 후보 선택
    tied_indices = [
        i for i, value in enumerate(candidate_f1)
        if np.isclose(value, best_f1)
    ]

    best_index = min(
        tied_indices,
        key=lambda i: abs(
            threshold_candidates[i] - common_threshold
        )
    )

    best_threshold = float(
        threshold_candidates[best_index]
    )

    per_function_thresholds.append(best_threshold)

    new_prediction = (
        scores >= best_threshold
    ).astype(int)

    old_prediction = (
        scores >= common_threshold
    ).astype(int)

    threshold_summary_rows.append({
        "기능": label,
        "기존 기준값": common_threshold,
        "새 기준값": best_threshold,
        "기존 검증 F1": f1_score(
            y_true, old_prediction, zero_division=0
        ),
        "새 검증 F1": best_f1,
        "새 Precision": precision_score(
            y_true, new_prediction, zero_division=0
        ),
        "새 Recall": recall_score(
            y_true, new_prediction, zero_division=0
        ),
        "검증 양성 개수": int(y_true.sum()),
        "양성 예측 수": int(new_prediction.sum())
    })

# 순서는 mlb.classes_와 동일
per_function_thresholds = np.array(
    per_function_thresholds
)

threshold_summary = pd.DataFrame(
    threshold_summary_rows
)

display(threshold_summary.round(4))

,기능,기존 기준값,새 기준값,기존 검증 F1,새 검증 F1,새 Precision,새 Recall,검증 양성 개수,양성 예측 수
0,SKIN CONDITIONING - HUMECTANT,-0.25,-0.50,0.2041,0.2597,0.2500,0.2703,37,40
1,SKIN CONDITIONING - EMOLLIENT,-0.25,-0.25,0.4930,0.4930,0.4910,0.4949,495,499
2,ANTI-SEBUM,-0.25,-0.25,0.3137,0.3137,0.4706,0.2353,34,17
3,SKIN PROTECTING,-0.25,-0.25,0.4136,0.4136,0.4520,0.3812,383,323
4,SKIN CONDITIONING - OCCLUSIVE,-0.25,-0.25,0.5000,0.5000,1.0000,0.3333,9,3
5,SOOTHING,-0.25,-0.50,0.2308,0.2857,0.2941,0.2778,18,17


### 공통 기준값과 기능별 기준값의 검증 성능 비교


In [52]:
# 기존 방식: 모든 기능에 같은 기준값
common_val_prediction = (
    svm_desc_val_scores >= common_threshold
).astype(int)

# 새 방식: 기능마다 다른 기준값
per_function_val_prediction = (
    svm_desc_val_scores >= per_function_thresholds
).astype(int)

threshold_comparison = pd.DataFrame([
    {
        "방식": "공통 기준값",
        **evaluate_predictions(
            y_val,
            common_val_prediction
        )
    },
    {
        "방식": "기능별 기준값",
        **evaluate_predictions(
            y_val,
            per_function_val_prediction
        )
    }
])

display(threshold_comparison.round(4))

,방식,Micro F1,Macro F1,Micro Precision,Micro Recall
0,공통 기준값,0.4461,0.3592,0.4756,0.4201
1,기능별 기준값,0.4448,0.3776,0.4638,0.4273


### 여섯 번째 문제 · SVM의 일반화 성능과 과적합

**여섯 번째 문제:** 기준값 조정만으로 성능을 크게 높이기 어려웠다.

**여섯 번째 해결:** C=0.1, 1, 10, 100에서 학습·검증 점수와 격차를 비교했다.

**여섯 번째 한계 — 실험 후 확인:** 큰 C는 학습 점수만 높였고 검증 성능은 낮아졌다. C=0.1은 검증 Micro F1 기준 선택이지만 개선 폭이 작고 C=1보다 Macro F1이 낮았다.


### SVM의 C 값에 따른 학습·검증 성능 비교


In [53]:
import time
import numpy as np

from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

c_candidates = [0.1, 1.0, 10.0, 100.0]

# 이전 SVM 기본 실험과 동일한 기준값 후보
threshold_candidates_c = [
    -0.9, -0.75, -0.5, -0.25, 0.0, 0.25
]

c_models = {}
c_results = {}
comparison_rows = []

for c_value in c_candidates:
    print(f"\nC={c_value} 학습 중...")

    model = OneVsRestClassifier(
        LinearSVC(
            C=c_value,
            max_iter=20000,
            random_state=SEED
        ),
        n_jobs=1
    )

    start_time = time.perf_counter()
    model.fit(X_train_desc, y_train)
    elapsed = time.perf_counter() - start_time

    train_scores = model.decision_function(X_train_desc)
    val_scores = model.decision_function(X_val_desc)

    # 검증 데이터에서 공통 기준값 선택
    threshold_rows = []

    for threshold in threshold_candidates_c:
        val_prediction = (
            val_scores >= threshold
        ).astype(int)

        threshold_rows.append({
            "기준값": threshold,
            **evaluate_predictions(y_val, val_prediction)
        })

    threshold_table = pd.DataFrame(threshold_rows)

    best_row = threshold_table.loc[
        threshold_table["Micro F1"].idxmax()
    ]

    best_threshold = float(best_row["기준값"])

    # 같은 기준값으로 학습·검증 성능 비교
    train_prediction = (
        train_scores >= best_threshold
    ).astype(int)

    val_prediction = (
        val_scores >= best_threshold
    ).astype(int)

    train_metrics = evaluate_predictions(
        y_train, train_prediction
    )
    val_metrics = evaluate_predictions(
        y_val, val_prediction
    )

    c_models[c_value] = model
    c_results[c_value] = {
        "threshold": best_threshold,
        "threshold_table": threshold_table,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "training_seconds": elapsed
    }

    comparison_rows.append({
        "C": c_value,
        "기준값": best_threshold,
        "학습 Micro F1": train_metrics["Micro F1"],
        "검증 Micro F1": val_metrics["Micro F1"],
        "학습-검증 차이": (
            train_metrics["Micro F1"]
            - val_metrics["Micro F1"]
        ),
        "검증 Macro F1": val_metrics["Macro F1"],
        "검증 Precision": val_metrics["Micro Precision"],
        "검증 Recall": val_metrics["Micro Recall"],
        "학습 시간(초)": elapsed
    })

    print(f"완료: {elapsed:.2f}초")

c_comparison = pd.DataFrame(comparison_rows)

display(c_comparison.round(4))

# 검증 Micro F1 기준으로 선택
best_c = max(
    c_results,
    key=lambda c: c_results[c]["val_metrics"]["Micro F1"]
)

print("검증 Micro F1 기준 선택 C:", best_c)
print("선택한 기준값:", c_results[best_c]["threshold"])


C=0.1 학습 중...
완료: 2.36초

C=1.0 학습 중...
완료: 4.49초

C=10.0 학습 중...
완료: 17.32초

C=100.0 학습 중...
완료: 137.23초


,C,기준값,학습 Micro F1,검증 Micro F1,학습-검증 차이,검증 Macro F1,검증 Precision,검증 Recall,학습 시간(초)
0,0.1,-0.50,0.6930,0.4493,0.2437,0.2504,0.4489,0.4498,2.3570
1,1.0,-0.25,0.9402,0.4461,0.4940,0.3592,0.4756,0.4201,4.4898
2,10.0,-0.25,0.9932,0.4153,0.5780,0.3402,0.3887,0.4457,17.3214
3,100.0,0.00,0.9995,0.3909,0.6086,0.3141,0.4024,0.3801,137.2267


검증 Micro F1 기준 선택 C: 0.1
선택한 기준값: -0.5


SVM의 C 값을 0.1, 1, 10, 100으로 비교하였다. C가 커질수록 학습 Micro F1은 증가했으나 검증 성능은 하락하여 과적합 경향을 확인하였다. 검증 Micro F1을 기준으로 C=0.1과 공통 기준값 -0.5를 선택하였다. 다만 C=1 대비 개선 폭은 작았으며, Macro F1은 낮아지는 상충 관계가 나타났다.

## 8. 최종 모델 평가와 저장

최종 배포 설정은 TF-IDF(성분명+설명), Linear SVM, C=0.1, 공통 결정 기준값 -0.5입니다. cosmetic_function_svm_final.joblib를 저장합니다.


### 최종 설정 고정 및 테스트 평가


In [56]:
# 최종 설정 고정
final_c = 0.1
final_model = c_models[final_c]
final_threshold = c_results[final_c]["threshold"]

# 성분명+설명 테스트 입력 생성
X_test_desc_text = make_description_input(test_data)

# 기존 TF-IDF 변환기 적용
X_test_desc = desc_vectorizer.transform(X_test_desc_text)

# 테스트 예측
final_test_scores = final_model.decision_function(X_test_desc)

final_test_pred = (
    final_test_scores >= final_threshold
).astype(int)

print("최종 모델: TF-IDF + Linear SVM")
print("입력: 성분명 + 화학적 설명")
print("C:", final_c)
print("기준값:", final_threshold)

# 전체 테스트 성능
final_test_metrics = evaluate_predictions(
    y_test, final_test_pred
)

display(
    pd.DataFrame(
        final_test_metrics.items(),
        columns=["평가지표", "점수"]
    ).round(4)
)

# 기능별 테스트 성능
final_function_table = function_metrics(y_test, final_test_pred)
display(final_function_table.round(4))


최종 모델: TF-IDF + Linear SVM
입력: 성분명 + 화학적 설명
C: 0.1
기준값: -0.5


,평가지표,점수
0,Micro F1,0.4567
1,Macro F1,0.1662
2,Micro Precision,0.4392
3,Micro Recall,0.4756


,precision,recall,f1-score,support,양성 예측 수
SKIN CONDITIONING - HUMECTANT,0.0000,0.0000,0.0000,21.0,2
SKIN CONDITIONING - EMOLLIENT,0.4730,0.6047,0.5308,506.0,647
ANTI-SEBUM,0.3333,0.0385,0.0690,52.0,6
SKIN PROTECTING,0.3846,0.4118,0.3977,340.0,364
SKIN CONDITIONING - OCCLUSIVE,0.0000,0.0000,0.0000,2.0,0
SOOTHING,0.0000,0.0000,0.0000,21.0,1


### 최종 모델과 평가 결과 저장


In [57]:
import joblib
import sklearn
import numpy as np
import scipy

from google.colab import files

final_bundle = {
    "model": final_model,
    "vectorizer": desc_vectorizer,
    "labels": list(mlb.classes_),
    "function_names": function_names,

    "threshold": float(final_threshold),
    "score_method": "decision_function",
    "input_type": "ingredient_name_and_description",
    "description_lookup": description_lookup,

    "normalization": "NFKC → 공백 통일 → 앞뒤 공백 제거 → 대문자",
    "seed": SEED,
    "C": final_c,

    "versions": {
        "sklearn": sklearn.__version__,
        "numpy": np.__version__,
        "scipy": scipy.__version__
    },

    "validation_metrics": c_results[final_c]["val_metrics"],
    "test_metrics": final_test_metrics,
    "evaluation_note": "동일 테스트셋을 사용한 후속 평가"
}

final_path = "cosmetic_function_svm_final.joblib"

joblib.dump(final_bundle, final_path, compress=3)

print("최종 모델 저장 완료:", final_path)

# 평가 결과도 별도 저장
final_function_table.to_csv(
    "final_function_test_results.csv",
    encoding="utf-8-sig"
)

# 내 컴퓨터로 다운로드
files.download(final_path)
files.download("final_function_test_results.csv")

최종 모델 저장 완료: cosmetic_function_svm_final.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. 모델 추론 확인과 초기 시연 사례

원본 등록 기능 조회와 모델 예측을 구분합니다. 검증 데이터에서 골라낸 성공 사례는 전체 성능을 대표하지 않습니다. 예시 버튼은 현재 서비스에서 제거했습니다.


### 일곱 번째 문제 · 최종 모델도 실제 입력에서 양성이 없을 수 있음

**일곱 번째 문제:** 익숙한 일부 성분에서 선택한 6개 기능의 양성 예측이 모두 없었다.

**일곱 번째 해결:** 저장한 모델의 로드·입력 변환·기준값 적용을 점검하고, 기능별 예측 사례와 등록 라벨을 비교했다. 서비스에서는 양성 없음과 조회 불가를 구분했다.

**일곱 번째 한계 — 실험 후 확인:** 선정한 성공 사례는 시연 확인용이며 성능 개선의 증거가 아니다. 음성 예측은 실제 기능이 없다는 확정이 아니며 추천 순위도 검증된 피부 적합도가 아니다.


### 최종 모델을 불러와 데모용 예측 함수 만들기


In [58]:
import joblib
import pandas as pd
import re
import unicodedata

# 최종 파일 불러오기
demo_bundle = joblib.load(
    "cosmetic_function_svm_final.joblib"
)

def normalize_ingredient_name(name):
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip().upper()


def predict_ingredient_demo(ingredient_name):
    name = normalize_ingredient_name(ingredient_name)

    if not name:
        return {
            "성분명": name,
            "상태": "성분명을 입력해 주세요.",
            "결과": None
        }

    # 저장된 설명 조회
    description = demo_bundle["description_lookup"].get(name)

    if description is None:
        return {
            "성분명": name,
            "상태": "등록된 설명을 찾지 못해 예측을 보류합니다.",
            "결과": None
        }

    if not description.strip():
        return {
            "성분명": name,
            "상태": "설명이 비어 있어 예측을 보류합니다.",
            "결과": None
        }

    # 학습 때와 동일한 입력 구성
    input_text = name + " " + description

    features = demo_bundle["vectorizer"].transform(
        [input_text]
    )

    scores = demo_bundle["model"].decision_function(
        features
    )[0]

    predictions = scores >= demo_bundle["threshold"]

    result = pd.DataFrame({
        "기능": demo_bundle["labels"],
        "설명": [
            demo_bundle["function_names"].get(label, label)
            for label in demo_bundle["labels"]
        ],
        "SVM 결정 점수": scores,
        "모델 예측": predictions.astype(int)
    })

    return {
        "성분명": name,
        "상태": "모델 예측 완료",
        "결과": result
    }


# 입력 예시
demo_result = predict_ingredient_demo("GLYCERIN")

print("성분명:", demo_result["성분명"])
print("상태:", demo_result["상태"])

if demo_result["결과"] is not None:
    display(demo_result["결과"].round(4))

성분명: GLYCERIN
상태: 모델 예측 완료


,기능,설명,SVM 결정 점수,모델 예측
0,SKIN CONDITIONING - HUMECTANT,피부 보습,-1.0839,0
1,SKIN CONDITIONING - EMOLLIENT,피부 유연화,-0.5037,0
2,ANTI-SEBUM,피지 조절,-1.0739,0
3,SKIN PROTECTING,피부 보호,-0.5728,0
4,SKIN CONDITIONING - OCCLUSIVE,수분 증발 억제,-0.9753,0
5,SOOTHING,피부·두피 불편감 완화,-1.0655,0


### 여러 성분 예측 및 결과 요약


In [59]:
def analyze_ingredients(ingredient_text):
    # 한 줄에 성분 하나씩 입력
    names = list(dict.fromkeys(
        normalize_ingredient_name(line)
        for line in ingredient_text.splitlines()
        if line.strip()
    ))

    if not names:
        raise ValueError("성분명을 한 개 이상 입력해 주세요.")

    summary_rows = []
    detail_tables = []

    for name in names:
        output = predict_ingredient_demo(name)
        result = output["결과"]

        if result is None:
            summary_rows.append({
                "성분명": name,
                "처리 상태": output["상태"],
                "모델이 예측한 기능": "예측 보류"
            })
            continue

        predicted_functions = result.loc[
            result["모델 예측"] == 1,
            "설명"
        ].tolist()

        summary_rows.append({
            "성분명": name,
            "처리 상태": "예측 완료",
            "모델이 예측한 기능": (
                ", ".join(predicted_functions)
                if predicted_functions
                else "선택한 6개 기능 중 양성 예측 없음"
            )
        })

        detail = result.copy()
        detail.insert(0, "성분명", name)
        detail_tables.append(detail)

    summary = pd.DataFrame(summary_rows)

    details = (
        pd.concat(detail_tables, ignore_index=True)
        if detail_tables
        else pd.DataFrame()
    )

    return summary, details


# 영문 INCI 성분명을 한 줄에 하나씩 입력
ingredient_text = """
GLYCERIN
NIACINAMIDE
SALICYLIC ACID
"""

demo_summary, demo_details = analyze_ingredients(
    ingredient_text
)

display(demo_summary)
display(demo_details.round(4))

,성분명,처리 상태,모델이 예측한 기능
0,GLYCERIN,예측 완료,선택한 6개 기능 중 양성 예측 없음
1,NIACINAMIDE,예측 완료,선택한 6개 기능 중 양성 예측 없음
2,SALICYLIC ACID,예측 완료,선택한 6개 기능 중 양성 예측 없음


,성분명,기능,설명,SVM 결정 점수,모델 예측
0,GLYCERIN,SKIN CONDITIONING - HUMECTANT,피부 보습,-1.0839,0
1,GLYCERIN,SKIN CONDITIONING - EMOLLIENT,피부 유연화,-0.5037,0
2,GLYCERIN,ANTI-SEBUM,피지 조절,-1.0739,0
3,GLYCERIN,SKIN PROTECTING,피부 보호,-0.5728,0
4,GLYCERIN,SKIN CONDITIONING - OCCLUSIVE,수분 증발 억제,-0.9753,0
5,GLYCERIN,SOOTHING,피부·두피 불편감 완화,-1.0655,0
6,NIACINAMIDE,SKIN CONDITIONING - HUMECTANT,피부 보습,-1.0129,0
7,NIACINAMIDE,SKIN CONDITIONING - EMOLLIENT,피부 유연화,-1.1627,0
8,NIACINAMIDE,ANTI-SEBUM,피지 조절,-1.0204,0
9,NIACINAMIDE,SKIN PROTECTING,피부 보호,-0.8381,0


### 기능별로 예측이 맞은 성분 찾기


In [60]:
import numpy as np

# 저장된 모델 기준으로 검증 입력 구성
example_names = val_data["INCI name"].map(
    normalize_ingredient_name
)

example_descriptions = example_names.map(
    demo_bundle["description_lookup"]
).fillna("")

example_texts = (
    example_names + " " + example_descriptions
)

example_features = demo_bundle["vectorizer"].transform(
    example_texts
)

example_scores = demo_bundle["model"].decision_function(
    example_features
)

example_predictions = (
    example_scores >= demo_bundle["threshold"]
)

example_rows = []

for column_index, label in enumerate(demo_bundle["labels"]):
    # 원본에 해당 기능이 기록되어 있는지 확인
    actual_positive = val_data["labels"].apply(
        lambda labels: label in labels
    ).to_numpy()

    # 설명이 있고, 모델 예측과 원본 기록이 모두 양성인 사례
    matched = (
        actual_positive
        & example_predictions[:, column_index]
        & example_descriptions.ne("").to_numpy()
    )

    matched_indices = np.flatnonzero(matched)

    # 점수가 높은 순으로 최대 3개 선택
    ranked_indices = sorted(
        matched_indices,
        key=lambda i: example_scores[i, column_index],
        reverse=True
    )[:3]

    for i in ranked_indices:
        example_rows.append({
            "기능": label,
            "설명": demo_bundle["function_names"].get(
                label, label
            ),
            "성분명": example_names.iloc[i],
            "SVM 결정 점수": example_scores[i, column_index],
            "원본 등록 여부": "기록됨",
            "모델 예측": 1
        })

demo_examples = pd.DataFrame(example_rows)

if demo_examples.empty:
    print("현재 조건에서 일치하는 사례가 없습니다.")
else:
    display(demo_examples.round(4))

    print("\n복사해서 입력할 수 있는 성분명:")
    print("\n".join(
        demo_examples["성분명"].drop_duplicates()
    ))

# 예측이 맞은 사례를 찾지 못한 기능도 표시
found_labels = {row["기능"] for row in example_rows}

missing_labels = [
    label
    for label in demo_bundle["labels"]
    if label not in found_labels
]

print("\n일치하는 사례를 찾지 못한 기능:")
print(missing_labels if missing_labels else "없음")

,기능,설명,성분명,SVM 결정 점수,원본 등록 여부,모델 예측
0,SKIN CONDITIONING - HUMECTANT,피부 보습,GINSENOSIDE RG1,-0.2945,기록됨,1
1,SKIN CONDITIONING - HUMECTANT,피부 보습,SILICA POLYSILSESQUIOXANE PROPYLDISULFIDEACETY...,-0.2959,기록됨,1
2,SKIN CONDITIONING - EMOLLIENT,피부 유연화,ISOSTEARYL BEHENATE,1.0862,기록됨,1
3,SKIN CONDITIONING - EMOLLIENT,피부 유연화,DIHEXYLDECYL SEBACATE,0.9816,기록됨,1
4,SKIN CONDITIONING - EMOLLIENT,피부 유연화,TRIHEXYLDECYL CITRATE,0.7347,기록됨,1
5,ANTI-SEBUM,피지 조절,R-(SH-OLIGOPEPTIDE-75 SH-POLYPEPTIDE-8),-0.4110,기록됨,1
6,ANTI-SEBUM,피지 조절,BACILLUS/MONASCUS/PRUNELLA VULGARIS FLOWER/LEA...,-0.4380,기록됨,1
7,SKIN PROTECTING,피부 보호,S-HIV-1 OLIGOPEPTIDE-6 SH-OLIGOPEPTIDE-1 SP,0.7895,기록됨,1
8,SKIN PROTECTING,피부 보호,SR-RAINBOW TROUT OLIGOPEPTIDE-1 SH-OLIGOPEPTIDE-2,0.6157,기록됨,1
9,SKIN PROTECTING,피부 보호,R-(S-RAINBOW TROUT OLIGOPEPTIDE-1 SH-POLYPEPTI...,0.5529,기록됨,1



복사해서 입력할 수 있는 성분명:
GINSENOSIDE RG1
SILICA POLYSILSESQUIOXANE PROPYLDISULFIDEACETYL PENTAPEPTIDE-4
ISOSTEARYL BEHENATE
DIHEXYLDECYL SEBACATE
TRIHEXYLDECYL CITRATE
R-(SH-OLIGOPEPTIDE-75 SH-POLYPEPTIDE-8)
BACILLUS/MONASCUS/PRUNELLA VULGARIS FLOWER/LEAF/SOYBEAN FERMENT FILTRATE
S-HIV-1 OLIGOPEPTIDE-6 SH-OLIGOPEPTIDE-1 SP
SR-RAINBOW TROUT OLIGOPEPTIDE-1 SH-OLIGOPEPTIDE-2
R-(S-RAINBOW TROUT OLIGOPEPTIDE-1 SH-POLYPEPTIDE-4)
ACETYLATED HYDROGENATED LARD GLYCERIDE
VISCUM ALBUM
ALLANTOIN POLYGALACTURONIC ACID

일치하는 사례를 찾지 못한 기능:
없음


### 프로토타입에 사용할 ‘등록 기능 조회 + AI 예측’ 연결


In [61]:
# 원본에서 성분명·기능 가져오기
lookup_data = df[["INCI name", "Function"]].copy()
lookup_data = lookup_data.dropna(subset=["INCI name"])

lookup_data["INCI name"] = (
    lookup_data["INCI name"].apply(normalize_ingredient_name)
)

# 빈 값과 NOT REPORTED 제외
lookup_data["registered_functions"] = (
    lookup_data["Function"]
    .fillna("")
    .apply(split_functions)
)

# 중복 성분명의 기능 통합
registered_lookup = (
    lookup_data
    .groupby("INCI name")["registered_functions"]
    .agg(
        lambda lists: sorted({
            label
            for labels in lists
            for label in labels
        })
    )
    .to_dict()
)

# 예시 버튼용 성분: 검증 데이터에서 선정한 시연 사례
sample_ingredients = (
    demo_examples["성분명"]
    .drop_duplicates()
    .tolist()
)

# 최종 파일에 추가
demo_bundle["registered_lookup"] = registered_lookup
demo_bundle["sample_ingredients"] = sample_ingredients
demo_bundle["sample_note"] = (
    "검증 데이터에서 기능 예측이 맞은 시연용 사례이며, "
    "전체 성능을 대표하지 않습니다."
)

joblib.dump(
    demo_bundle,
    "cosmetic_function_svm_final.joblib",
    compress=3
)

print("등록 기능과 예시 성분 저장 완료!")

등록 기능과 예시 성분 저장 완료!


### 성분 목록을 분석하는 공통 함수


In [62]:
def analyze_product_ingredients(ingredient_names):
    names = list(dict.fromkeys(
        normalize_ingredient_name(name)
        for name in ingredient_names
        if name.strip()
    ))

    if not names:
        raise ValueError("성분명을 한 개 이상 입력해 주세요.")

    rows = []

    for name in names:
        registered = demo_bundle["registered_lookup"].get(name)
        output = predict_ingredient_demo(name)

        # 등록 정보 조회
        if registered is None:
            registered_text = "원본에서 조회되지 않음"
        elif not registered:
            registered_text = "보고된 기능 없음"
        else:
            registered_text = ", ".join(registered)

        # AI 예측
        if output["결과"] is None:
            predicted_text = "예측 보류"
        else:
            predicted_functions = output["결과"].loc[
                output["결과"]["모델 예측"] == 1,
                "설명"
            ].tolist()

            predicted_text = (
                ", ".join(predicted_functions)
                if predicted_functions
                else "선택한 6개 기능 중 양성 예측 없음"
            )

        rows.append({
            "성분명": name,
            "CosIng 등록 기능": registered_text,
            "AI 예측 기능": predicted_text,
            "처리 상태": output["상태"]
        })

    return pd.DataFrame(rows)

### 실제 입력과 예시 입력 테스트


In [63]:
# 실제 제품 입력을 가정한 테스트
product_ingredients = [
    "GLYCERIN",
    "NIACINAMIDE",
    "SALICYLIC ACID"
]

print("제품 성분 분석")
display(
    analyze_product_ingredients(product_ingredients)
)

# 예시 버튼을 눌렀을 때 사용할 입력
print("\nAI 동작 확인용 예시")
print(demo_bundle["sample_note"])

display(
    analyze_product_ingredients(
        demo_bundle["sample_ingredients"][:3]
    )
)

# 업데이트한 파일 다시 다운로드
files.download("cosmetic_function_svm_final.joblib")

제품 성분 분석


,성분명,CosIng 등록 기능,AI 예측 기능,처리 상태
0,GLYCERIN,"DENATURANT, HAIR CONDITIONING, HUMECTANT, ORAL...",선택한 6개 기능 중 양성 예측 없음,모델 예측 완료
1,NIACINAMIDE,SMOOTHING,선택한 6개 기능 중 양성 예측 없음,모델 예측 완료
2,SALICYLIC ACID,"ANTI-SEBORRHEIC, FRAGRANCE, HAIR CONDITIONING,...",선택한 6개 기능 중 양성 예측 없음,모델 예측 완료



AI 동작 확인용 예시
검증 데이터에서 기능 예측이 맞은 시연용 사례이며, 전체 성능을 대표하지 않습니다.


,성분명,CosIng 등록 기능,AI 예측 기능,처리 상태
0,GINSENOSIDE RG1,"HAIR CONDITIONING, SKIN CONDITIONING - HUMECTANT",피부 보습,모델 예측 완료
1,SILICA POLYSILSESQUIOXANE PROPYLDISULFIDEACETY...,"SKIN CONDITIONING - EMOLLIENT, SKIN CONDITIONI...",피부 보습,모델 예측 완료
2,ISOSTEARYL BEHENATE,"SKIN CONDITIONING, SKIN CONDITIONING - EMOLLIENT",피부 유연화,모델 예측 완료


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. 부록: 초기 배포 코드 생성 기록

아래는 처음 Vercel 배포를 준비했던 코드와 출력 기록입니다. 재실행하면 초기 UI와 API가 생성됩니다. 최종 제출에는 현재 서비스 소스를 별도 파일로 함께 포함하세요.


### Vercel 배포 전 최종 모델 파일 확인


In [64]:
from pathlib import Path
import joblib
import sklearn
import numpy as np
import scipy

model_path = Path("cosmetic_function_svm_final.joblib")

if not model_path.exists():
    raise FileNotFoundError(
        "최종 모델 파일을 Colab에 업로드해 주세요."
    )

deployment_bundle = joblib.load(model_path)

print("모델 파일:", model_path.name)
print(f"파일 크기: {model_path.stat().st_size / 1024**2:.2f} MB")
print("모델 종류:", type(deployment_bundle["model"]).__name__)
print("예측 기준값:", deployment_bundle["threshold"])
print("기능 수:", len(deployment_bundle["labels"]))
print(
    "설명 조회 가능한 성분 수:",
    len(deployment_bundle["description_lookup"])
)

print("\n현재 라이브러리 버전")
print("scikit-learn:", sklearn.__version__)
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("joblib:", joblib.__version__)

print("\n학습 시 저장한 버전")
print(deployment_bundle.get("versions", "저장된 버전 정보 없음"))

모델 파일: cosmetic_function_svm_final.joblib
파일 크기: 4.67 MB
모델 종류: OneVsRestClassifier
예측 기준값: -0.5
기능 수: 6
설명 조회 가능한 성분 수: 30079

현재 라이브러리 버전
scikit-learn: 1.6.1
numpy: 2.4.6
scipy: 1.16.3
joblib: 1.6.0

학습 시 저장한 버전
{'sklearn': '1.6.1', 'numpy': '2.4.6', 'scipy': '1.16.3'}


### 배포용 폴더 준비


In [65]:
from pathlib import Path
import shutil

project_dir = Path("cosmetic-demo")
(project_dir / "api").mkdir(parents=True, exist_ok=True)

shutil.copy2(
    "cosmetic_function_svm_final.joblib",
    project_dir / "cosmetic_function_svm_final.joblib"
)

requirements = """scikit-learn==1.6.1
numpy==2.4.6
scipy==1.16.3
joblib==1.6.0
"""

(project_dir / "requirements.txt").write_text(
    requirements,
    encoding="utf-8"
)

print("배포 폴더 준비 완료!")

배포 폴더 준비 완료!


### 성분명을 받아 모델을 실행하는 코드 만들기


In [66]:
api_code = r'''
import json
import re
import unicodedata
from http.server import BaseHTTPRequestHandler
from pathlib import Path

import joblib

ROOT = Path(__file__).resolve().parents[1]
bundle = joblib.load(ROOT / "cosmetic_function_svm_final.joblib")


def normalize_name(name):
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip().upper()


def analyze(names):
    names = list(dict.fromkeys(
        normalize_name(name) for name in names if name.strip()
    ))

    results = []

    for name in names:
        description = bundle["description_lookup"].get(name)

        if not description or not description.strip():
            results.append({
                "ingredient": name,
                "status": "説明情報がありません",
                "message": "설명을 찾지 못해 예측을 보류합니다.",
                "functions": []
            })
            continue

        text = name + " " + description
        features = bundle["vectorizer"].transform([text])
        scores = bundle["model"].decision_function(features)[0]

        functions = []

        for label, score in zip(bundle["labels"], scores):
            functions.append({
                "label": label,
                "name": bundle["function_names"].get(label, label),
                "score": float(score),
                "predicted": bool(score >= bundle["threshold"])
            })

        results.append({
            "ingredient": name,
            "status": "predicted",
            "message": "학습한 SVM으로 예측했습니다.",
            "functions": functions
        })

    return results


class handler(BaseHTTPRequestHandler):
    def send_json(self, status, data):
        body = json.dumps(data, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        self.send_json(200, {
            "message": "화장품 성분 기능 예측 API",
            "model": "TF-IDF + Linear SVM",
            "threshold": float(bundle["threshold"]),
            "samples": bundle.get("sample_ingredients", []),
            "sample_note": "검증 데이터에서 선정한 시연용 사례"
        })

    def do_POST(self):
        try:
            length = int(self.headers.get("Content-Length", "0"))

            if length <= 0 or length > 50000:
                self.send_json(400, {"error": "입력 크기를 확인해 주세요."})
                return

            payload = json.loads(self.rfile.read(length))
            names = payload.get("ingredients") if isinstance(payload, dict) else None

            if (
                not isinstance(names, list)
                or not 1 <= len(names) <= 100
                or not all(
                    isinstance(name, str) and 0 < len(name.strip()) <= 3000
                    for name in names
                )
            ):
                self.send_json(400, {
                    "error": "성분명을 문자열 목록으로 1~100개 입력해 주세요."
                })
                return

            self.send_json(200, {"results": analyze(names)})

        except (ValueError, UnicodeDecodeError):
            self.send_json(400, {"error": "입력 형식을 확인해 주세요."})
        except Exception:
            self.send_json(500, {"error": "모델 예측 처리 중 오류가 발생했습니다."})
'''

# 상태값 통일
api_code = api_code.replace(
    '"status": "説明情報がありません"',
    '"status": "description_unavailable"'
)

(project_dir / "api" / "predict.py").write_text(
    api_code,
    encoding="utf-8"
)

print("api/predict.py 생성 완료!")

api/predict.py 생성 완료!


### 배포용 코드가 기존 함수와 같은 결과를 내는지 확인


In [67]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "demo_predict_api",
    project_dir / "api" / "predict.py"
)

api_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(api_module)

# 예시 성분이 저장되어 있는지 확인
samples = api_module.bundle.get("sample_ingredients", [])

print("저장된 예시 성분 수:", len(samples))

test_names = ["GLYCERIN"] + samples[:2]
api_results = api_module.analyze(test_names)

for result in api_results:
    predicted = [
        function["name"]
        for function in result["functions"]
        if function["predicted"]
    ]

    print("\n성분:", result["ingredient"])
    print("상태:", result["message"])
    print("예측 기능:", predicted or "양성 예측 없음")

저장된 예시 성분 수: 13

성분: GLYCERIN
상태: 학습한 SVM으로 예측했습니다.
예측 기능: 양성 예측 없음

성분: GINSENOSIDE RG1
상태: 학습한 SVM으로 예측했습니다.
예측 기능: ['피부 보습']

성분: SILICA POLYSILSESQUIOXANE PROPYLDISULFIDEACETYL PENTAPEPTIDE-4
상태: 학습한 SVM으로 예측했습니다.
예측 기능: ['피부 보습']


### 성분 입력창·예시 버튼·결과 화면 만들기


In [68]:
html_code = r'''
<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>화장품 성분 기능 분석</title>
<style>
* { box-sizing: border-box; }
body {
  margin: 0; background: #f5f7fa; color: #202936;
  font-family: system-ui, sans-serif;
}
main { max-width: 850px; margin: 50px auto; padding: 24px; }
h1 { font-size: 28px; }
p { line-height: 1.7; }
.panel {
  background: white; padding: 24px; border-radius: 16px;
  margin-top: 20px; border: 1px solid #e2e7ee;
}
label { display: block; font-weight: 600; margin-bottom: 10px; }
textarea {
  width: 100%; min-height: 180px; padding: 14px;
  border: 1px solid #bbc5d1; border-radius: 10px;
  font: inherit; resize: vertical;
}
.actions { display: flex; flex-wrap: wrap; gap: 10px; margin-top: 14px; }
button {
  padding: 12px 18px; border-radius: 9px; border: 1px solid #cad3df;
  background: white; font: inherit; cursor: pointer;
}
.primary { background: #2357c6; color: white; border-color: #2357c6; }
button:disabled { opacity: .6; cursor: wait; }
.note { color: #576579; font-size: 14px; }
.result { padding: 18px 0; border-bottom: 1px solid #e2e7ee; }
.result:last-child { border-bottom: 0; }
.result h3 { overflow-wrap: anywhere; }
.badge {
  display: inline-block; background: #eaf1ff; color: #214a93;
  padding: 7px 11px; margin: 4px; border-radius: 20px;
}
#status { min-height: 24px; }
</style>
</head>
<body>
<main>
  <h1>화장품 성분 기능 분석</h1>
  <p>
    직접 학습한 TF-IDF + Linear SVM 모델로
    성분의 6가지 기능을 예측하는 실험용 데모입니다.
  </p>

  <section class="panel">
    <label for="ingredients">영문 INCI 성분명</label>
    <textarea id="ingredients"
      placeholder="한 줄에 성분 하나씩 입력하세요.&#10;GLYCERIN&#10;NIACINAMIDE"></textarea>

    <div class="actions">
      <button id="analyze" class="primary">성분 분석하기</button>
      <button id="samples">시연용 예시 불러오기</button>
    </div>

    <p class="note">
      예시는 검증 데이터에서 예측이 맞은 사례로 선정했으며,
      전체 성능을 대표하지 않습니다.
    </p>
    <p id="status" role="status" aria-live="polite"></p>
  </section>

  <section id="results" class="panel" hidden>
    <h2>학습한 SVM의 예측 결과</h2>
    <p class="note">
      결과는 성분 기능의 추정이며, 개인 피부 적합도나 실제 효능의 판정이 아닙니다.
      양성 예측이 없어도 기능이 전혀 없다는 의미는 아닙니다.
    </p>
    <div id="result-list"></div>
  </section>
</main>

<script>
const input = document.getElementById("ingredients");
const status = document.getElementById("status");
const analyzeButton = document.getElementById("analyze");
const sampleButton = document.getElementById("samples");

sampleButton.addEventListener("click", async () => {
  sampleButton.disabled = true;
  status.textContent = "예시 성분을 불러오는 중입니다.";

  try {
    const response = await fetch("/api/predict");
    if (!response.ok) throw new Error("예시를 불러오지 못했습니다.");

    const data = await response.json();
    if (!data.samples?.length) throw new Error("저장된 예시가 없습니다.");

    input.value = data.samples.join("\n");
    status.textContent = "예시를 불러왔습니다. 성분 분석하기를 눌러 주세요.";
  } catch (error) {
    status.textContent = error.message;
  } finally {
    sampleButton.disabled = false;
  }
});

analyzeButton.addEventListener("click", async () => {
  const ingredients = [...new Set(
    input.value.split("\n").map(value => value.trim()).filter(Boolean)
  )];

  if (!ingredients.length || ingredients.length > 100) {
    status.textContent = "성분명을 한 줄에 하나씩, 1~100개 입력해 주세요.";
    return;
  }

  analyzeButton.disabled = true;
  document.getElementById("results").hidden = true;
  status.textContent = "저장된 모델로 예측하는 중입니다.";

  try {
    const response = await fetch("/api/predict", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ ingredients })
    });

    const data = await response.json();
    if (!response.ok) throw new Error(data.error || "분석에 실패했습니다.");

    const list = document.getElementById("result-list");
    list.replaceChildren();

    for (const result of data.results) {
      const article = document.createElement("article");
      article.className = "result";

      const title = document.createElement("h3");
      title.textContent = result.ingredient;
      article.appendChild(title);

      const predicted = result.functions.filter(item => item.predicted);

      if (result.status !== "predicted" || !predicted.length) {
        const message = document.createElement("p");
        message.textContent = result.status !== "predicted"
          ? result.message
          : "선택한 6개 기능 중 양성 예측 없음";
        article.appendChild(message);
      } else {
        for (const item of predicted) {
          const badge = document.createElement("span");
          badge.className = "badge";
          badge.textContent = item.name;
          article.appendChild(badge);
        }
      }

      list.appendChild(article);
    }

    document.getElementById("results").hidden = false;
    status.textContent = `${data.results.length}개 성분 분석 완료`;
  } catch (error) {
    status.textContent = error.message;
  } finally {
    analyzeButton.disabled = false;
  }
});
</script>
</body>
</html>
'''

(project_dir / "index.html").write_text(
    html_code,
    encoding="utf-8"
)

print("웹 화면 생성 완료!")

웹 화면 생성 완료!


### 배포 파일 다운로드


In [69]:
import shutil
from google.colab import files

zip_path = shutil.make_archive(
    "cosmetic-demo",
    "zip",
    root_dir=project_dir
)

files.download(zip_path)

print("다운로드 완료! ZIP 압축을 풀어 주세요.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

다운로드 완료! ZIP 압축을 풀어 주세요.


In [70]:
print(
    (project_dir / "api" / "predict.py").read_text(
        encoding="utf-8"
    )
)


import json
import re
import unicodedata
from http.server import BaseHTTPRequestHandler
from pathlib import Path

import joblib

ROOT = Path(__file__).resolve().parents[1]
bundle = joblib.load(ROOT / "cosmetic_function_svm_final.joblib")


def normalize_name(name):
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip().upper()


def analyze(names):
    names = list(dict.fromkeys(
        normalize_name(name) for name in names if name.strip()
    ))

    results = []

    for name in names:
        description = bundle["description_lookup"].get(name)

        if not description or not description.strip():
            results.append({
                "ingredient": name,
                "status": "description_unavailable",
                "message": "설명을 찾지 못해 예측을 보류합니다.",
                "functions": []
            })
            continue

        text = name + " " + description
        features = bundle["vectorizer"].transform([text])
        scores